First version. Only valid ISBNs obtained by removing invalid characters (but result not checked for actual existence). Only non-zero ratings. Only users and books that have at least 3 ratings. 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import gc
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix, csr_matrix

from load_data import load_data
from valid_test_select import valid_test_select
from initialize_model import initialize_mu_b_c
from helpers import get_rmse
from fit_model import fit_model_no_UV, fit_model_full

In [3]:
np.set_printoptions(linewidth=150, precision=6)  # 75, 8

# Loading Data

In [4]:
df = load_data()

In [5]:
df.agg(["min", "max", "nunique", "dtype", "count", "size"])

,userid,isbn,rating
min,2,0000001481,0
max,278854,9998150752,10
nunique,103371,328465,11
dtype,int32,object,int8
count,1135338,1135338,1135338
size,1135338,1135338,1135338


In [6]:
df = df[df.rating > 0]

In [7]:
df.agg(["min", "max", "nunique", "size"])

,userid,isbn,rating
min,8,000003827X,1
max,278854,9997555635,10
nunique,76331,179924,10
size,427261,427261,427261


# Creating Matrix Y

In [8]:
# Encoding userids and isbns as categories (integers 0 to N-1)

user_cats = df.userid.astype("category")
book_cats = df.isbn.astype("category")

In [9]:
# Creating sparse matrix and converting it to pd.DataFrame

Y_sparse = coo_matrix((df.rating.values, (user_cats.cat.codes, book_cats.cat.codes)))
Y = pd.DataFrame(Y_sparse.toarray())
del Y_sparse
gc.collect()

40

In [10]:
# print("Y shape:", Y.shape)  # (76331, 179924)
# print("total entries:", (Y > 0).sum().sum())  # 427261
# print("avg number of ratings per user:", round((Y > 0).sum(axis=1).mean(), 2))  # 5.6
# print("avg number of ratings per book:", round((Y > 0).sum(axis=0).mean(), 2))  # 2.37

In [11]:
# Filtering out improbably active readers

# np.sort(np.partition((Y > 0).sum(axis=1), -20)[-20:])
Y = Y.loc[(Y > 0).sum(axis=1) < 2000, :]

In [12]:
# Filtering users and books with enough observations

min_rats = 3
old_rows, old_cols = Y.shape
while True:
    Y_pos = Y > 0  # type: ignore
    user_rats = Y_pos.sum(axis=1)  # type: ignore
    book_rats = Y_pos.sum(axis=0)  # type: ignore
    new_rows = (user_rats >= min_rats).sum()
    new_cols = (book_rats >= min_rats).sum()
    # print(new_rows, new_cols)
    Y = Y.loc[user_rats >= min_rats, book_rats >= min_rats]  # type: ignore
    if (old_rows == new_rows) and (old_cols == new_cols):
        break
    old_rows, old_cols = new_rows, new_cols

del Y_pos, user_rats, book_rats, new_rows, new_cols, old_rows, old_cols
gc.collect()

0

In [13]:
# Converting Y to numpy array

Y_columns = Y.columns.copy()
Y = Y.to_numpy()

In [14]:
def get_isbn(pos):
    """Given position of column in matrix Y, return the original ISBN."""
    return book_cats.cat.categories[Y_columns[pos]]

# Calculating "p-values"

In [15]:
to_try = [
    # [1, 4.36666666666667, 24.3333333333333],
    # [2, 4.33333333333333, 22.6666666666667],
    # [4, 4.33333333333333, 25.5],
    # [8, 4.33333333333333, 22.6666666666667],
    # [16, 4.3, 20.7333333333333],
    # [32, 4.2, 14.3333333333333],
    # [64, 4.2, 13.5],
    # [128, 4.13333333333333, 11.6666666666667],
    # [256, 4.03333333333333, 10.6666666666667],
    [384, 3.9666666666666663, 9.666666666666667]
]

In [16]:
results_biases = np.zeros(shape=(1, 100))
results_full = np.zeros(shape=(len(to_try), 100))

In [17]:
for seed in range(1000, 1100):
    print(f"{seed = }")
    # Selecting validation and test sets
    Y_train, valid_data, test_data = valid_test_select(Y, 20_000, 20_000, seed=seed)
    Y_csr = csr_matrix(Y_train)
    Y_csc = Y_csr.tocsc()

    # Model with only biases
    print("    only biases:", end=" ")
    mu, b, c = initialize_mu_b_c(Y_train)
    mu, b, c, _ = fit_model_no_UV(Y_csr, Y_csc, mu, b, c, 4.4, valid_data, 1e-5, info=1)

    preds = np.clip(mu + b[test_data.rows] + c[test_data.cols], 1, 10)
    results_biases[0, seed - 1000] = get_rmse(preds, test_data)

    # Full model
    i = 0
    for k, l_b, l_f in to_try:
        print(f"    {k = :>3}", end=",     ")
        mu, b, c = initialize_mu_b_c(Y_train)
        rng = np.random.default_rng(seed=123)
        U = rng.normal(0, 0.5, size=(Y.shape[0], k))
        V = rng.normal(0, 0.5, size=(Y.shape[1], k))
        mu, b, c, U, V, _ = fit_model_full(
            Y_csr, Y_csc, mu, b, c, U, V, l_b, l_f, valid_data, 1e-5, info=1
        )

        biases = mu + b[test_data.rows] + c[test_data.cols]
        preds = np.clip(
            biases + np.sum(U[test_data.rows] * V[test_data.cols], axis=1), 1, 10
        )
        # "i" was "round(np.log2(k))"
        results_full[i, seed - 1000] = get_rmse(preds, test_data)
        i += 1

seed = 1000
    only biases: results: counter =  17, max_norm_diff =   0.06, valid_rmse = 1.55810142, valid_rmse_diff = 7.8e-06
    k = 384,     results: counter =   8, max_norm_diff =   3.66, valid_rmse = 1.55528767, valid_rmse_diff = 3.7e-05
seed = 1001
    only biases: results: counter =  17, max_norm_diff =   0.06, valid_rmse = 1.57181952, valid_rmse_diff = 7.9e-06
    k = 384,     results: counter =  10, max_norm_diff =   2.76, valid_rmse = 1.57086868, valid_rmse_diff = 1.4e-05
seed = 1002
    only biases: results: counter =  15, max_norm_diff =   0.10, valid_rmse = 1.56414289, valid_rmse_diff = 8.4e-06
    k = 384,     results: counter =   5, max_norm_diff =   6.66, valid_rmse = 1.56159760, valid_rmse_diff = 0.00016
seed = 1003
    only biases: results: counter =  14, max_norm_diff =   0.13, valid_rmse = 1.55487124, valid_rmse_diff = 8.8e-06
    k = 384,     results: counter =   5, max_norm_diff =   6.65, valid_rmse = 1.55424561, valid_rmse_diff = 3.4e-05
seed = 1004
    only bia

In [18]:
assert not (results_biases == 0).sum()
assert not (results_full == 0).sum()

In [19]:
(results_full >= results_biases).sum(axis=1)
# array([38, 38,  2,  5,  0, 13,  5,  5,  1])

array([1])

In [20]:
# seed = 1000
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5581, valid_rmse_diff = 7.8e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.13, valid_rmse = 1.5575, valid_rmse_diff = 6.5e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.15, valid_rmse = 1.5577, valid_rmse_diff = 9.5e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5577, valid_rmse_diff = 9.9e-06
#     k =   8,     results: counter =  25, max_norm_diff = 0.17, valid_rmse = 1.5575, valid_rmse_diff = 9.6e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.26, valid_rmse = 1.5576, valid_rmse_diff = 8.1e-06
#     k =  32,     results: counter =  28, max_norm_diff = 0.43, valid_rmse = 1.5565, valid_rmse_diff = 9.4e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.79, valid_rmse = 1.5556, valid_rmse_diff = 8.9e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5552, valid_rmse_diff = 9.8e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.17, valid_rmse = 1.5554, valid_rmse_diff = 3.3e-06
# seed = 1001
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5718, valid_rmse_diff = 7.9e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5716, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  24, max_norm_diff = 0.11, valid_rmse = 1.5712, valid_rmse_diff = 9.2e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.17, valid_rmse = 1.5717, valid_rmse_diff = 9.9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.23, valid_rmse = 1.5715, valid_rmse_diff = 8.7e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.30, valid_rmse = 1.5714, valid_rmse_diff = 8.6e-06
#     k =  32,     results: counter =  26, max_norm_diff = 0.50, valid_rmse = 1.5715, valid_rmse_diff = 9.8e-06
#     k =  64,     results: counter =  17, max_norm_diff = 0.89, valid_rmse = 1.5714, valid_rmse_diff = 9.7e-06
#     k = 128,     results: counter =  24, max_norm_diff = 0.68, valid_rmse = 1.5714, valid_rmse_diff = 8.7e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.94, valid_rmse = 1.5709, valid_rmse_diff = 6.6e-06
# seed = 1002
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5641, valid_rmse_diff = 8.4e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.30, valid_rmse = 1.5640, valid_rmse_diff = 8.5e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.31, valid_rmse = 1.5639, valid_rmse_diff = 7e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5640, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  12, max_norm_diff = 0.28, valid_rmse = 1.5642, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.40, valid_rmse = 1.5637, valid_rmse_diff = 6.6e-06
#     k =  32,     results: counter =   7, max_norm_diff = 2.29, valid_rmse = 1.5637, valid_rmse_diff = 1.7e-05
#     k =  64,     results: counter =  10, max_norm_diff = 1.72, valid_rmse = 1.5637, valid_rmse_diff = 9.5e-07
#     k = 128,     results: counter =   7, max_norm_diff = 3.35, valid_rmse = 1.5623, valid_rmse_diff = 5.1e-05
#     k = 256,     results: counter =   5, max_norm_diff = 5.87, valid_rmse = 1.5622, valid_rmse_diff = 0.00016
# seed = 1003
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5549, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  11, max_norm_diff = 0.48, valid_rmse = 1.5547, valid_rmse_diff = 8.9e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.31, valid_rmse = 1.5543, valid_rmse_diff = 8.2e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.18, valid_rmse = 1.5547, valid_rmse_diff = 7.7e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5546, valid_rmse_diff = 8.6e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.41, valid_rmse = 1.5544, valid_rmse_diff = 3.9e-06
#     k =  32,     results: counter =   4, max_norm_diff = 5.26, valid_rmse = 1.5554, valid_rmse_diff = 2.9e-05
#     k =  64,     results: counter =  11, max_norm_diff = 1.55, valid_rmse = 1.5543, valid_rmse_diff = 1.1e-05
#     k = 128,     results: counter =   7, max_norm_diff = 3.32, valid_rmse = 1.5539, valid_rmse_diff = 2.1e-05
#     k = 256,     results: counter =   6, max_norm_diff = 4.58, valid_rmse = 1.5536, valid_rmse_diff = 4e-05
# seed = 1004
#     only biases: results: counter =  14, max_norm_diff = 0.12, valid_rmse = 1.5662, valid_rmse_diff = 9.6e-06
#     k =   1,     results: counter =   8, max_norm_diff = 0.53, valid_rmse = 1.5661, valid_rmse_diff = 1e-05
#     k =   2,     results: counter =   6, max_norm_diff = 1.41, valid_rmse = 1.5666, valid_rmse_diff = 3.1e-05
#     k =   4,     results: counter =  14, max_norm_diff = 0.18, valid_rmse = 1.5659, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.19, valid_rmse = 1.5659, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.34, valid_rmse = 1.5657, valid_rmse_diff = 1e-05
#     k =  32,     results: counter =   9, max_norm_diff = 1.68, valid_rmse = 1.5654, valid_rmse_diff = 6.4e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.07, valid_rmse = 1.5652, valid_rmse_diff = 7e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5640, valid_rmse_diff = 3.5e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.80, valid_rmse = 1.5636, valid_rmse_diff = 1.2e-05
# seed = 1005
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5582, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  12, max_norm_diff = 0.40, valid_rmse = 1.5579, valid_rmse_diff = 5.3e-06
#     k =   2,     results: counter =   6, max_norm_diff = 1.22, valid_rmse = 1.5583, valid_rmse_diff = 2.6e-05
#     k =   4,     results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5579, valid_rmse_diff = 8.6e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.20, valid_rmse = 1.5577, valid_rmse_diff = 9e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.40, valid_rmse = 1.5575, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  35, max_norm_diff = 0.34, valid_rmse = 1.5571, valid_rmse_diff = 9.8e-06
#     k =  64,     results: counter =  20, max_norm_diff = 0.74, valid_rmse = 1.5565, valid_rmse_diff = 7.1e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.20, valid_rmse = 1.5557, valid_rmse_diff = 2.6e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.46, valid_rmse = 1.5562, valid_rmse_diff = 1.1e-05
# seed = 1006
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5472, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5468, valid_rmse_diff = 9.8e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.25, valid_rmse = 1.5469, valid_rmse_diff = 6.4e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.18, valid_rmse = 1.5468, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.17, valid_rmse = 1.5465, valid_rmse_diff = 8.2e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5464, valid_rmse_diff = 8.7e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.19, valid_rmse = 1.5442, valid_rmse_diff = 9.4e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.86, valid_rmse = 1.5447, valid_rmse_diff = 7.9e-06
#     k = 128,     results: counter =  18, max_norm_diff = 1.05, valid_rmse = 1.5444, valid_rmse_diff = 6.2e-06
#     k = 256,     results: counter =  14, max_norm_diff = 1.62, valid_rmse = 1.5436, valid_rmse_diff = 1.9e-06
# seed = 1007
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5799, valid_rmse_diff = 8.4e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.15, valid_rmse = 1.5798, valid_rmse_diff = 9.6e-06
#     k =   2,     results: counter =  18, max_norm_diff = 0.18, valid_rmse = 1.5795, valid_rmse_diff = 9.4e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5797, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.20, valid_rmse = 1.5795, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.29, valid_rmse = 1.5795, valid_rmse_diff = 6.3e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.89, valid_rmse = 1.5776, valid_rmse_diff = 8.2e-06
#     k =  64,     results: counter =  20, max_norm_diff = 0.72, valid_rmse = 1.5778, valid_rmse_diff = 9.2e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.44, valid_rmse = 1.5766, valid_rmse_diff = 4.6e-06
#     k = 256,     results: counter =  14, max_norm_diff = 1.58, valid_rmse = 1.5767, valid_rmse_diff = 6.4e-06
# seed = 1008
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5688, valid_rmse_diff = 8.9e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.12, valid_rmse = 1.5683, valid_rmse_diff = 8.6e-06
#     k =   2,     results: counter =  32, max_norm_diff = 0.16, valid_rmse = 1.5680, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5685, valid_rmse_diff = 8.3e-06
#     k =   8,     results: counter =  21, max_norm_diff = 0.15, valid_rmse = 1.5684, valid_rmse_diff = 9.4e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.28, valid_rmse = 1.5683, valid_rmse_diff = 9.7e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.14, valid_rmse = 1.5667, valid_rmse_diff = 4e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.09, valid_rmse = 1.5666, valid_rmse_diff = 8.8e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.20, valid_rmse = 1.5659, valid_rmse_diff = 6.2e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.22, valid_rmse = 1.5650, valid_rmse_diff = 3.1e-05
# seed = 1009
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5585, valid_rmse_diff = 9.9e-06
#     k =   1,     results: counter =  11, max_norm_diff = 0.40, valid_rmse = 1.5586, valid_rmse_diff = 4e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.24, valid_rmse = 1.5581, valid_rmse_diff = 9.1e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5582, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.23, valid_rmse = 1.5580, valid_rmse_diff = 7.3e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.34, valid_rmse = 1.5578, valid_rmse_diff = 9.4e-06
#     k =  32,     results: counter =  19, max_norm_diff = 0.74, valid_rmse = 1.5568, valid_rmse_diff = 8.1e-06
#     k =  64,     results: counter =  17, max_norm_diff = 0.90, valid_rmse = 1.5566, valid_rmse_diff = 9.1e-06
#     k = 128,     results: counter =  20, max_norm_diff = 0.89, valid_rmse = 1.5561, valid_rmse_diff = 9.8e-06
#     k = 256,     results: counter =  13, max_norm_diff = 1.76, valid_rmse = 1.5559, valid_rmse_diff = 7e-06
# seed = 1010
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5505, valid_rmse_diff = 9.3e-06
#     k =   1,     results: counter =  12, max_norm_diff = 0.27, valid_rmse = 1.5502, valid_rmse_diff = 8.5e-06
#     k =   2,     results: counter =  11, max_norm_diff = 0.37, valid_rmse = 1.5504, valid_rmse_diff = 7.9e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5502, valid_rmse_diff = 7.4e-06
#     k =   8,     results: counter =  12, max_norm_diff = 0.28, valid_rmse = 1.5503, valid_rmse_diff = 8.2e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.40, valid_rmse = 1.5503, valid_rmse_diff = 5.2e-06
#     k =  32,     results: counter =  43, max_norm_diff = 0.24, valid_rmse = 1.5497, valid_rmse_diff = 9.8e-06
#     k =  64,     results: counter =  21, max_norm_diff = 0.69, valid_rmse = 1.5494, valid_rmse_diff = 9.4e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.19, valid_rmse = 1.5487, valid_rmse_diff = 1.3e-05
#     k = 256,     results: counter =  10, max_norm_diff = 2.44, valid_rmse = 1.5485, valid_rmse_diff = 6.4e-06
# seed = 1011
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5606, valid_rmse_diff = 9.4e-06
#     k =   1,     results: counter =   6, max_norm_diff = 1.20, valid_rmse = 1.5608, valid_rmse_diff = 5.9e-05
#     k =   2,     results: counter =  12, max_norm_diff = 0.34, valid_rmse = 1.5601, valid_rmse_diff = 3.2e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5603, valid_rmse_diff = 6.7e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.19, valid_rmse = 1.5601, valid_rmse_diff = 8.8e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.25, valid_rmse = 1.5599, valid_rmse_diff = 1e-05
#     k =  32,     results: counter =  10, max_norm_diff = 1.57, valid_rmse = 1.5600, valid_rmse_diff = 3.2e-06
#     k =  64,     results: counter =  11, max_norm_diff = 1.55, valid_rmse = 1.5603, valid_rmse_diff = 8.3e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.45, valid_rmse = 1.5594, valid_rmse_diff = 3.7e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.79, valid_rmse = 1.5592, valid_rmse_diff = 2.2e-05
# seed = 1012
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5677, valid_rmse_diff = 8.4e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5674, valid_rmse_diff = 9.9e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5675, valid_rmse_diff = 8.8e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5673, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5673, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  27, max_norm_diff = 0.15, valid_rmse = 1.5670, valid_rmse_diff = 9.6e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.89, valid_rmse = 1.5667, valid_rmse_diff = 4.2e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.07, valid_rmse = 1.5673, valid_rmse_diff = 5.8e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.45, valid_rmse = 1.5671, valid_rmse_diff = 3.5e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.46, valid_rmse = 1.5669, valid_rmse_diff = 1.4e-05
# seed = 1013
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5571, valid_rmse_diff = 9e-06
#     k =   1,     results: counter =   7, max_norm_diff = 1.42, valid_rmse = 1.5574, valid_rmse_diff = 5.5e-06
#     k =   2,     results: counter =   6, max_norm_diff = 1.20, valid_rmse = 1.5576, valid_rmse_diff = 2.2e-05
#     k =   4,     results: counter =  13, max_norm_diff = 0.17, valid_rmse = 1.5570, valid_rmse_diff = 1e-05
#     k =   8,     results: counter =  14, max_norm_diff = 0.18, valid_rmse = 1.5567, valid_rmse_diff = 6.6e-06
#     k =  16,     results: counter =  23, max_norm_diff = 0.20, valid_rmse = 1.5569, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  29, max_norm_diff = 0.44, valid_rmse = 1.5562, valid_rmse_diff = 9.3e-06
#     k =  64,     results: counter =  32, max_norm_diff = 0.36, valid_rmse = 1.5567, valid_rmse_diff = 9.9e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.95, valid_rmse = 1.5558, valid_rmse_diff = 1.8e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5549, valid_rmse_diff = 3.4e-05
# seed = 1014
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5641, valid_rmse_diff = 8.6e-06
#     k =   1,     results: counter =   9, max_norm_diff = 0.44, valid_rmse = 1.5642, valid_rmse_diff = 3e-05
#     k =   2,     results: counter =  14, max_norm_diff = 0.25, valid_rmse = 1.5637, valid_rmse_diff = 6.7e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.11, valid_rmse = 1.5638, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.25, valid_rmse = 1.5637, valid_rmse_diff = 7.2e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.28, valid_rmse = 1.5638, valid_rmse_diff = 7.3e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.38, valid_rmse = 1.5634, valid_rmse_diff = 8.4e-06
#     k =  64,     results: counter =  11, max_norm_diff = 1.54, valid_rmse = 1.5633, valid_rmse_diff = 2.8e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.74, valid_rmse = 1.5634, valid_rmse_diff = 5e-07
#     k = 256,     results: counter =   9, max_norm_diff = 2.77, valid_rmse = 1.5630, valid_rmse_diff = 7e-06
# seed = 1015
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5581, valid_rmse_diff = 8.9e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.12, valid_rmse = 1.5574, valid_rmse_diff = 8.8e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.18, valid_rmse = 1.5571, valid_rmse_diff = 8.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.11, valid_rmse = 1.5575, valid_rmse_diff = 8e-06
#     k =   8,     results: counter =  26, max_norm_diff = 0.14, valid_rmse = 1.5571, valid_rmse_diff = 9.5e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.25, valid_rmse = 1.5571, valid_rmse_diff = 1e-05
#     k =  32,     results: counter =  15, max_norm_diff = 0.96, valid_rmse = 1.5566, valid_rmse_diff = 8.3e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.17, valid_rmse = 1.5570, valid_rmse_diff = 8.3e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.47, valid_rmse = 1.5568, valid_rmse_diff = 2.1e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.77, valid_rmse = 1.5564, valid_rmse_diff = 5e-05
# seed = 1016
#     only biases: results: counter =  12, max_norm_diff = 0.20, valid_rmse = 1.5610, valid_rmse_diff = 6.7e-06
#     k =   1,     results: counter =  11, max_norm_diff = 0.55, valid_rmse = 1.5607, valid_rmse_diff = 3.7e-06
#     k =   2,     results: counter =  23, max_norm_diff = 0.19, valid_rmse = 1.5604, valid_rmse_diff = 9e-06
#     k =   4,     results: counter =  12, max_norm_diff = 0.21, valid_rmse = 1.5607, valid_rmse_diff = 6.5e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.21, valid_rmse = 1.5605, valid_rmse_diff = 9.4e-06
#     k =  16,     results: counter =  13, max_norm_diff = 0.37, valid_rmse = 1.5602, valid_rmse_diff = 5.8e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.87, valid_rmse = 1.5610, valid_rmse_diff = 7.9e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.84, valid_rmse = 1.5608, valid_rmse_diff = 8.8e-06
#     k = 128,     results: counter =   7, max_norm_diff = 3.37, valid_rmse = 1.5598, valid_rmse_diff = 1.6e-06
#     k = 256,     results: counter =   6, max_norm_diff = 4.60, valid_rmse = 1.5601, valid_rmse_diff = 3.6e-05
# seed = 1017
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5659, valid_rmse_diff = 8.6e-06
#     k =   1,     results: counter =   9, max_norm_diff = 0.49, valid_rmse = 1.5658, valid_rmse_diff = 4.6e-05
#     k =   2,     results: counter =  18, max_norm_diff = 0.13, valid_rmse = 1.5657, valid_rmse_diff = 9.4e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.12, valid_rmse = 1.5657, valid_rmse_diff = 6.6e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5653, valid_rmse_diff = 7.2e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.26, valid_rmse = 1.5653, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  10, max_norm_diff = 1.53, valid_rmse = 1.5646, valid_rmse_diff = 4.5e-06
#     k =  64,     results: counter =  30, max_norm_diff = 0.42, valid_rmse = 1.5650, valid_rmse_diff = 9.4e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.32, valid_rmse = 1.5647, valid_rmse_diff = 5.4e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.77, valid_rmse = 1.5639, valid_rmse_diff = 1.9e-05
# seed = 1018
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5668, valid_rmse_diff = 8.9e-06
#     k =   1,     results: counter =  13, max_norm_diff = 0.59, valid_rmse = 1.5666, valid_rmse_diff = 7.5e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5663, valid_rmse_diff = 7.8e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.16, valid_rmse = 1.5664, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.21, valid_rmse = 1.5662, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5661, valid_rmse_diff = 8.1e-06
#     k =  32,     results: counter =  10, max_norm_diff = 1.50, valid_rmse = 1.5659, valid_rmse_diff = 4.5e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.25, valid_rmse = 1.5658, valid_rmse_diff = 9.6e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.44, valid_rmse = 1.5646, valid_rmse_diff = 7.5e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5640, valid_rmse_diff = 1.4e-05
# seed = 1019
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5475, valid_rmse_diff = 9.8e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.15, valid_rmse = 1.5472, valid_rmse_diff = 7.7e-06
#     k =   2,     results: counter =  11, max_norm_diff = 0.39, valid_rmse = 1.5473, valid_rmse_diff = 2.6e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.13, valid_rmse = 1.5472, valid_rmse_diff = 7.8e-06
#     k =   8,     results: counter =  13, max_norm_diff = 0.26, valid_rmse = 1.5474, valid_rmse_diff = 7.4e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.38, valid_rmse = 1.5473, valid_rmse_diff = 9.1e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.25, valid_rmse = 1.5470, valid_rmse_diff = 9.3e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.85, valid_rmse = 1.5476, valid_rmse_diff = 8.2e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.51, valid_rmse = 1.5480, valid_rmse_diff = 1.1e-05
#     k = 256,     results: counter =   6, max_norm_diff = 4.65, valid_rmse = 1.5474, valid_rmse_diff = 5.5e-05
# seed = 1020
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5601, valid_rmse_diff = 9.4e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.17, valid_rmse = 1.5596, valid_rmse_diff = 9.4e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.17, valid_rmse = 1.5597, valid_rmse_diff = 8.6e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5598, valid_rmse_diff = 8.8e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5595, valid_rmse_diff = 9.2e-06
#     k =  16,     results: counter =  23, max_norm_diff = 0.19, valid_rmse = 1.5596, valid_rmse_diff = 9.8e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.97, valid_rmse = 1.5577, valid_rmse_diff = 6.4e-06
#     k =  64,     results: counter =  21, max_norm_diff = 0.70, valid_rmse = 1.5585, valid_rmse_diff = 9.5e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5579, valid_rmse_diff = 1.4e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.75, valid_rmse = 1.5571, valid_rmse_diff = 3.5e-06
# seed = 1021
#     only biases: results: counter =  19, max_norm_diff = 0.04, valid_rmse = 1.5656, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.08, valid_rmse = 1.5654, valid_rmse_diff = 7.8e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.21, valid_rmse = 1.5653, valid_rmse_diff = 9e-06
#     k =   4,     results: counter =  20, max_norm_diff = 0.11, valid_rmse = 1.5653, valid_rmse_diff = 8.2e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.20, valid_rmse = 1.5652, valid_rmse_diff = 8.7e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.29, valid_rmse = 1.5652, valid_rmse_diff = 7.3e-06
#     k =  32,     results: counter =  24, max_norm_diff = 0.59, valid_rmse = 1.5648, valid_rmse_diff = 7.8e-06
#     k =  64,     results: counter =  21, max_norm_diff = 0.69, valid_rmse = 1.5647, valid_rmse_diff = 8.8e-06
#     k = 128,     results: counter =  17, max_norm_diff = 1.11, valid_rmse = 1.5643, valid_rmse_diff = 5.8e-06
#     k = 256,     results: counter =  14, max_norm_diff = 1.60, valid_rmse = 1.5644, valid_rmse_diff = 9.3e-06
# seed = 1022
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5676, valid_rmse_diff = 7.6e-06
#     k =   1,     results: counter =  24, max_norm_diff = 0.23, valid_rmse = 1.5671, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.25, valid_rmse = 1.5672, valid_rmse_diff = 8.3e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.15, valid_rmse = 1.5671, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  21, max_norm_diff = 0.17, valid_rmse = 1.5669, valid_rmse_diff = 9.9e-06
#     k =  16,     results: counter =  20, max_norm_diff = 0.24, valid_rmse = 1.5665, valid_rmse_diff = 8.3e-06
#     k =  32,     results: counter =  17, max_norm_diff = 0.86, valid_rmse = 1.5659, valid_rmse_diff = 5.6e-06
#     k =  64,     results: counter =  16, max_norm_diff = 0.98, valid_rmse = 1.5662, valid_rmse_diff = 7.3e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.93, valid_rmse = 1.5658, valid_rmse_diff = 8e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.76, valid_rmse = 1.5653, valid_rmse_diff = 7.9e-08
# seed = 1023
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5618, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =   6, max_norm_diff = 0.97, valid_rmse = 1.5621, valid_rmse_diff = 7.1e-07
#     k =   2,     results: counter =  15, max_norm_diff = 0.25, valid_rmse = 1.5613, valid_rmse_diff = 8.5e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.13, valid_rmse = 1.5616, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  10, max_norm_diff = 0.41, valid_rmse = 1.5613, valid_rmse_diff = 8e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.33, valid_rmse = 1.5612, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  27, max_norm_diff = 0.49, valid_rmse = 1.5601, valid_rmse_diff = 9.2e-06
#     k =  64,     results: counter =  20, max_norm_diff = 0.74, valid_rmse = 1.5599, valid_rmse_diff = 8e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5590, valid_rmse_diff = 9.8e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.46, valid_rmse = 1.5588, valid_rmse_diff = 1.2e-06
# seed = 1024
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5514, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =   7, max_norm_diff = 0.74, valid_rmse = 1.5519, valid_rmse_diff = 1e-05
#     k =   2,     results: counter =   8, max_norm_diff = 0.72, valid_rmse = 1.5518, valid_rmse_diff = 4.2e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.11, valid_rmse = 1.5512, valid_rmse_diff = 8.8e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5509, valid_rmse_diff = 7.2e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5511, valid_rmse_diff = 9.8e-06
#     k =  32,     results: counter =  17, max_norm_diff = 0.84, valid_rmse = 1.5514, valid_rmse_diff = 9.4e-06
#     k =  64,     results: counter =   5, max_norm_diff = 4.10, valid_rmse = 1.5513, valid_rmse_diff = 1.9e-05
#     k = 128,     results: counter =   9, max_norm_diff = 2.47, valid_rmse = 1.5504, valid_rmse_diff = 1.4e-05
#     k = 256,     results: counter =   6, max_norm_diff = 4.58, valid_rmse = 1.5500, valid_rmse_diff = 5.8e-05
# seed = 1025
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5680, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.25, valid_rmse = 1.5677, valid_rmse_diff = 7.9e-06
#     k =   2,     results: counter =  11, max_norm_diff = 0.44, valid_rmse = 1.5678, valid_rmse_diff = 6.9e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.16, valid_rmse = 1.5677, valid_rmse_diff = 8.3e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5676, valid_rmse_diff = 8.3e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.44, valid_rmse = 1.5677, valid_rmse_diff = 8.7e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.09, valid_rmse = 1.5675, valid_rmse_diff = 6.3e-06
#     k =  64,     results: counter =  27, max_norm_diff = 0.47, valid_rmse = 1.5675, valid_rmse_diff = 9.5e-06
#     k = 128,     results: counter =   8, max_norm_diff = 2.82, valid_rmse = 1.5662, valid_rmse_diff = 2.6e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.77, valid_rmse = 1.5668, valid_rmse_diff = 2.5e-05
# seed = 1026
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5652, valid_rmse_diff = 9.2e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.17, valid_rmse = 1.5651, valid_rmse_diff = 9e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.29, valid_rmse = 1.5650, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.11, valid_rmse = 1.5648, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.22, valid_rmse = 1.5646, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.30, valid_rmse = 1.5651, valid_rmse_diff = 7.9e-06
#     k =  32,     results: counter =  24, max_norm_diff = 0.53, valid_rmse = 1.5644, valid_rmse_diff = 7.3e-06
#     k =  64,     results: counter =  17, max_norm_diff = 0.92, valid_rmse = 1.5636, valid_rmse_diff = 7.4e-06
#     k = 128,     results: counter =  16, max_norm_diff = 1.21, valid_rmse = 1.5637, valid_rmse_diff = 7.6e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.96, valid_rmse = 1.5631, valid_rmse_diff = 1.1e-05
# seed = 1027
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5567, valid_rmse_diff = 7.9e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.16, valid_rmse = 1.5563, valid_rmse_diff = 7.6e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.18, valid_rmse = 1.5562, valid_rmse_diff = 8.3e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5563, valid_rmse_diff = 9.8e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5561, valid_rmse_diff = 9.4e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.25, valid_rmse = 1.5559, valid_rmse_diff = 9.4e-06
#     k =  32,     results: counter =  21, max_norm_diff = 0.66, valid_rmse = 1.5543, valid_rmse_diff = 7.8e-06
#     k =  64,     results: counter =  22, max_norm_diff = 0.64, valid_rmse = 1.5547, valid_rmse_diff = 9.4e-06
#     k = 128,     results: counter =  21, max_norm_diff = 0.83, valid_rmse = 1.5544, valid_rmse_diff = 9.4e-06
#     k = 256,     results: counter =  13, max_norm_diff = 1.76, valid_rmse = 1.5542, valid_rmse_diff = 7.2e-06
# seed = 1028
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5617, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5613, valid_rmse_diff = 8.1e-06
#     k =   2,     results: counter =  21, max_norm_diff = 0.17, valid_rmse = 1.5616, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5613, valid_rmse_diff = 8.8e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.17, valid_rmse = 1.5612, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5608, valid_rmse_diff = 8.6e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.37, valid_rmse = 1.5607, valid_rmse_diff = 1.6e-05
#     k =  64,     results: counter =  18, max_norm_diff = 0.82, valid_rmse = 1.5604, valid_rmse_diff = 6.9e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.31, valid_rmse = 1.5602, valid_rmse_diff = 8.1e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.95, valid_rmse = 1.5601, valid_rmse_diff = 2.3e-06
# seed = 1029
#     only biases: results: counter =  19, max_norm_diff = 0.04, valid_rmse = 1.5683, valid_rmse_diff = 8.3e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5680, valid_rmse_diff = 8.6e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.24, valid_rmse = 1.5681, valid_rmse_diff = 9.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.13, valid_rmse = 1.5680, valid_rmse_diff = 1e-05
#     k =   8,     results: counter =  20, max_norm_diff = 0.20, valid_rmse = 1.5678, valid_rmse_diff = 7.5e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.33, valid_rmse = 1.5681, valid_rmse_diff = 6.9e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.05, valid_rmse = 1.5672, valid_rmse_diff = 2.2e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.78, valid_rmse = 1.5670, valid_rmse_diff = 9e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.59, valid_rmse = 1.5664, valid_rmse_diff = 8.4e-06
#     k = 256,     results: counter =  13, max_norm_diff = 1.75, valid_rmse = 1.5673, valid_rmse_diff = 8.2e-06
# seed = 1030
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5642, valid_rmse_diff = 9.7e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.06, valid_rmse = 1.5639, valid_rmse_diff = 9.1e-06
#     k =   2,     results: counter =  31, max_norm_diff = 0.15, valid_rmse = 1.5638, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5637, valid_rmse_diff = 8.7e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.16, valid_rmse = 1.5636, valid_rmse_diff = 9.2e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.27, valid_rmse = 1.5633, valid_rmse_diff = 7.8e-06
#     k =  32,     results: counter =  23, max_norm_diff = 0.59, valid_rmse = 1.5629, valid_rmse_diff = 9.1e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.78, valid_rmse = 1.5628, valid_rmse_diff = 9.1e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.97, valid_rmse = 1.5627, valid_rmse_diff = 1.9e-05
#     k = 256,     results: counter =   9, max_norm_diff = 2.79, valid_rmse = 1.5631, valid_rmse_diff = 1.3e-05
# seed = 1031
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5652, valid_rmse_diff = 7.8e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.09, valid_rmse = 1.5646, valid_rmse_diff = 7.7e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.34, valid_rmse = 1.5650, valid_rmse_diff = 9.1e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5648, valid_rmse_diff = 8.8e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.19, valid_rmse = 1.5647, valid_rmse_diff = 9e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.25, valid_rmse = 1.5645, valid_rmse_diff = 8.7e-06
#     k =  32,     results: counter =  15, max_norm_diff = 1.00, valid_rmse = 1.5636, valid_rmse_diff = 3.4e-06
#     k =  64,     results: counter =  16, max_norm_diff = 0.99, valid_rmse = 1.5634, valid_rmse_diff = 9.3e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.44, valid_rmse = 1.5633, valid_rmse_diff = 6.9e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.45, valid_rmse = 1.5625, valid_rmse_diff = 1.2e-05
# seed = 1032
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5602, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.24, valid_rmse = 1.5600, valid_rmse_diff = 6.8e-06
#     k =   2,     results: counter =  13, max_norm_diff = 0.21, valid_rmse = 1.5601, valid_rmse_diff = 7.8e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5600, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.22, valid_rmse = 1.5598, valid_rmse_diff = 9.6e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.30, valid_rmse = 1.5593, valid_rmse_diff = 7.3e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.23, valid_rmse = 1.5595, valid_rmse_diff = 7.3e-06
#     k =  64,     results: counter =  12, max_norm_diff = 1.38, valid_rmse = 1.5593, valid_rmse_diff = 6.8e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.20, valid_rmse = 1.5585, valid_rmse_diff = 5.6e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5582, valid_rmse_diff = 1.4e-05
# seed = 1033
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5624, valid_rmse_diff = 9.6e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5621, valid_rmse_diff = 5.6e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.19, valid_rmse = 1.5622, valid_rmse_diff = 7.5e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.15, valid_rmse = 1.5621, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.19, valid_rmse = 1.5620, valid_rmse_diff = 8.6e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.35, valid_rmse = 1.5619, valid_rmse_diff = 6.4e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.27, valid_rmse = 1.5622, valid_rmse_diff = 7.5e-06
#     k =  64,     results: counter =   8, max_norm_diff = 2.24, valid_rmse = 1.5619, valid_rmse_diff = 2.1e-05
#     k = 128,     results: counter =   7, max_norm_diff = 3.38, valid_rmse = 1.5611, valid_rmse_diff = 2.9e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.79, valid_rmse = 1.5620, valid_rmse_diff = 4.9e-06
# seed = 1034
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5688, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5685, valid_rmse_diff = 7.3e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.21, valid_rmse = 1.5684, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.15, valid_rmse = 1.5686, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.15, valid_rmse = 1.5685, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.26, valid_rmse = 1.5683, valid_rmse_diff = 8.4e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.13, valid_rmse = 1.5681, valid_rmse_diff = 7.4e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.16, valid_rmse = 1.5676, valid_rmse_diff = 8.2e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.32, valid_rmse = 1.5671, valid_rmse_diff = 9.8e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.79, valid_rmse = 1.5663, valid_rmse_diff = 2.5e-05
# seed = 1035
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5640, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  21, max_norm_diff = 0.11, valid_rmse = 1.5634, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.22, valid_rmse = 1.5637, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.09, valid_rmse = 1.5637, valid_rmse_diff = 8.3e-06
#     k =   8,     results: counter =  21, max_norm_diff = 0.15, valid_rmse = 1.5636, valid_rmse_diff = 9.3e-06
#     k =  16,     results: counter =  21, max_norm_diff = 0.22, valid_rmse = 1.5633, valid_rmse_diff = 9.7e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.96, valid_rmse = 1.5631, valid_rmse_diff = 9e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.78, valid_rmse = 1.5629, valid_rmse_diff = 9e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.98, valid_rmse = 1.5618, valid_rmse_diff = 9.3e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.80, valid_rmse = 1.5613, valid_rmse_diff = 2.2e-05
# seed = 1036
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5466, valid_rmse_diff = 8.9e-06
#     k =   1,     results: counter =  20, max_norm_diff = 0.10, valid_rmse = 1.5462, valid_rmse_diff = 9.3e-06
#     k =   2,     results: counter =  18, max_norm_diff = 0.20, valid_rmse = 1.5460, valid_rmse_diff = 9.4e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.12, valid_rmse = 1.5462, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5459, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  22, max_norm_diff = 0.21, valid_rmse = 1.5455, valid_rmse_diff = 9.3e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.91, valid_rmse = 1.5448, valid_rmse_diff = 7.5e-06
#     k =  64,     results: counter =  16, max_norm_diff = 0.98, valid_rmse = 1.5449, valid_rmse_diff = 7.3e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.45, valid_rmse = 1.5448, valid_rmse_diff = 6e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.80, valid_rmse = 1.5441, valid_rmse_diff = 1.5e-05
# seed = 1037
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5608, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5606, valid_rmse_diff = 8.8e-06
#     k =   2,     results: counter =  11, max_norm_diff = 0.38, valid_rmse = 1.5606, valid_rmse_diff = 5.2e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.16, valid_rmse = 1.5606, valid_rmse_diff = 9.9e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.21, valid_rmse = 1.5605, valid_rmse_diff = 1e-05
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5607, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.15, valid_rmse = 1.5610, valid_rmse_diff = 4.3e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.25, valid_rmse = 1.5607, valid_rmse_diff = 1.7e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.18, valid_rmse = 1.5603, valid_rmse_diff = 1.8e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.78, valid_rmse = 1.5602, valid_rmse_diff = 1.4e-05
# seed = 1038
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5755, valid_rmse_diff = 8.2e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.33, valid_rmse = 1.5755, valid_rmse_diff = 9.9e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.16, valid_rmse = 1.5752, valid_rmse_diff = 8.9e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.11, valid_rmse = 1.5754, valid_rmse_diff = 9.7e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.20, valid_rmse = 1.5752, valid_rmse_diff = 9.5e-06
#     k =  16,     results: counter =  13, max_norm_diff = 0.38, valid_rmse = 1.5755, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.07, valid_rmse = 1.5766, valid_rmse_diff = 2.6e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.17, valid_rmse = 1.5767, valid_rmse_diff = 9.5e-06
#     k = 128,     results: counter =  16, max_norm_diff = 1.21, valid_rmse = 1.5765, valid_rmse_diff = 6e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.20, valid_rmse = 1.5761, valid_rmse_diff = 9e-06
# seed = 1039
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5551, valid_rmse_diff = 9.9e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.10, valid_rmse = 1.5549, valid_rmse_diff = 9e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.34, valid_rmse = 1.5544, valid_rmse_diff = 6.7e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.10, valid_rmse = 1.5548, valid_rmse_diff = 9.1e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5547, valid_rmse_diff = 9.2e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.27, valid_rmse = 1.5544, valid_rmse_diff = 7.9e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.22, valid_rmse = 1.5530, valid_rmse_diff = 6.7e-06
#     k =  64,     results: counter =  30, max_norm_diff = 0.40, valid_rmse = 1.5541, valid_rmse_diff = 9.6e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.19, valid_rmse = 1.5524, valid_rmse_diff = 9.5e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.80, valid_rmse = 1.5525, valid_rmse_diff = 2.3e-05
# seed = 1040
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5478, valid_rmse_diff = 8.9e-06
#     k =   1,     results: counter =  12, max_norm_diff = 0.33, valid_rmse = 1.5475, valid_rmse_diff = 1e-05
#     k =   2,     results: counter =  10, max_norm_diff = 0.38, valid_rmse = 1.5475, valid_rmse_diff = 7.9e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.15, valid_rmse = 1.5473, valid_rmse_diff = 8.5e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.19, valid_rmse = 1.5471, valid_rmse_diff = 8.5e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.40, valid_rmse = 1.5470, valid_rmse_diff = 7e-06
#     k =  32,     results: counter =   9, max_norm_diff = 1.76, valid_rmse = 1.5452, valid_rmse_diff = 2.7e-05
#     k =  64,     results: counter =  15, max_norm_diff = 1.09, valid_rmse = 1.5451, valid_rmse_diff = 5.6e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.45, valid_rmse = 1.5445, valid_rmse_diff = 6.7e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.22, valid_rmse = 1.5440, valid_rmse_diff = 1.1e-06
# seed = 1041
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5562, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.17, valid_rmse = 1.5560, valid_rmse_diff = 7.4e-06
#     k =   2,     results: counter =  25, max_norm_diff = 0.13, valid_rmse = 1.5559, valid_rmse_diff = 9.5e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5559, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.27, valid_rmse = 1.5558, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.26, valid_rmse = 1.5558, valid_rmse_diff = 8e-06
#     k =  32,     results: counter =  18, max_norm_diff = 0.78, valid_rmse = 1.5550, valid_rmse_diff = 6.8e-06
#     k =  64,     results: counter =  23, max_norm_diff = 0.60, valid_rmse = 1.5553, valid_rmse_diff = 9e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.42, valid_rmse = 1.5549, valid_rmse_diff = 4.9e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.22, valid_rmse = 1.5547, valid_rmse_diff = 3.1e-05
# seed = 1042
#     only biases: results: counter =  13, max_norm_diff = 0.16, valid_rmse = 1.5664, valid_rmse_diff = 9e-06
#     k =   1,     results: counter =   6, max_norm_diff = 0.86, valid_rmse = 1.5666, valid_rmse_diff = 2.7e-05
#     k =   2,     results: counter =  14, max_norm_diff = 0.21, valid_rmse = 1.5658, valid_rmse_diff = 7.3e-06
#     k =   4,     results: counter =  12, max_norm_diff = 0.22, valid_rmse = 1.5661, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5659, valid_rmse_diff = 8.3e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.39, valid_rmse = 1.5659, valid_rmse_diff = 4.7e-06
#     k =  32,     results: counter =  26, max_norm_diff = 0.49, valid_rmse = 1.5646, valid_rmse_diff = 9.4e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.83, valid_rmse = 1.5641, valid_rmse_diff = 8.7e-06
#     k = 128,     results: counter =  16, max_norm_diff = 1.18, valid_rmse = 1.5637, valid_rmse_diff = 8e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.92, valid_rmse = 1.5630, valid_rmse_diff = 5.4e-06
# seed = 1043
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5623, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =   8, max_norm_diff = 0.50, valid_rmse = 1.5623, valid_rmse_diff = 1.1e-05
#     k =   2,     results: counter =  20, max_norm_diff = 0.10, valid_rmse = 1.5617, valid_rmse_diff = 9.7e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.14, valid_rmse = 1.5618, valid_rmse_diff = 8.6e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5614, valid_rmse_diff = 8.9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.28, valid_rmse = 1.5615, valid_rmse_diff = 7.8e-06
#     k =  32,     results: counter =  43, max_norm_diff = 0.27, valid_rmse = 1.5614, valid_rmse_diff = 9.7e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.28, valid_rmse = 1.5601, valid_rmse_diff = 9.3e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.94, valid_rmse = 1.5597, valid_rmse_diff = 1.3e-05
#     k = 256,     results: counter =  11, max_norm_diff = 2.17, valid_rmse = 1.5602, valid_rmse_diff = 4.5e-06
# seed = 1044
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5537, valid_rmse_diff = 8.3e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5532, valid_rmse_diff = 7.5e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.12, valid_rmse = 1.5532, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5533, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.17, valid_rmse = 1.5530, valid_rmse_diff = 9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5528, valid_rmse_diff = 9.3e-06
#     k =  32,     results: counter =  20, max_norm_diff = 0.69, valid_rmse = 1.5512, valid_rmse_diff = 9.1e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.81, valid_rmse = 1.5506, valid_rmse_diff = 6.4e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.96, valid_rmse = 1.5502, valid_rmse_diff = 1.3e-05
#     k = 256,     results: counter =  10, max_norm_diff = 2.46, valid_rmse = 1.5508, valid_rmse_diff = 1.4e-05
# seed = 1045
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5616, valid_rmse_diff = 9.6e-06
#     k =   1,     results: counter =  21, max_norm_diff = 0.15, valid_rmse = 1.5611, valid_rmse_diff = 8.3e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.13, valid_rmse = 1.5613, valid_rmse_diff = 8.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.11, valid_rmse = 1.5612, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  22, max_norm_diff = 0.16, valid_rmse = 1.5609, valid_rmse_diff = 8.6e-06
#     k =  16,     results: counter =  20, max_norm_diff = 0.23, valid_rmse = 1.5604, valid_rmse_diff = 9.9e-06
#     k =  32,     results: counter =  44, max_norm_diff = 0.24, valid_rmse = 1.5594, valid_rmse_diff = 9.5e-06
#     k =  64,     results: counter =  16, max_norm_diff = 0.98, valid_rmse = 1.5588, valid_rmse_diff = 5.7e-06
#     k = 128,     results: counter =  18, max_norm_diff = 1.04, valid_rmse = 1.5591, valid_rmse_diff = 8e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.96, valid_rmse = 1.5588, valid_rmse_diff = 9.4e-06
# seed = 1046
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5483, valid_rmse_diff = 7.9e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.16, valid_rmse = 1.5478, valid_rmse_diff = 7.2e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.22, valid_rmse = 1.5479, valid_rmse_diff = 9.1e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.11, valid_rmse = 1.5479, valid_rmse_diff = 8.2e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.26, valid_rmse = 1.5478, valid_rmse_diff = 6.6e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.31, valid_rmse = 1.5477, valid_rmse_diff = 7.7e-06
#     k =  32,     results: counter =  22, max_norm_diff = 0.63, valid_rmse = 1.5465, valid_rmse_diff = 5.1e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.15, valid_rmse = 1.5465, valid_rmse_diff = 5.4e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.94, valid_rmse = 1.5459, valid_rmse_diff = 1.4e-05
#     k = 256,     results: counter =  10, max_norm_diff = 2.44, valid_rmse = 1.5461, valid_rmse_diff = 6.8e-06
# seed = 1047
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5656, valid_rmse_diff = 9.8e-06
#     k =   1,     results: counter =   8, max_norm_diff = 0.69, valid_rmse = 1.5658, valid_rmse_diff = 1.1e-05
#     k =   2,     results: counter =   7, max_norm_diff = 0.95, valid_rmse = 1.5661, valid_rmse_diff = 8.9e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.13, valid_rmse = 1.5655, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.20, valid_rmse = 1.5653, valid_rmse_diff = 8.5e-06
#     k =  16,     results: counter =  27, max_norm_diff = 0.16, valid_rmse = 1.5655, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.16, valid_rmse = 1.5645, valid_rmse_diff = 1.8e-06
#     k =  64,     results: counter =  12, max_norm_diff = 1.38, valid_rmse = 1.5648, valid_rmse_diff = 4.9e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.19, valid_rmse = 1.5646, valid_rmse_diff = 6.1e-07
#     k = 256,     results: counter =   9, max_norm_diff = 2.79, valid_rmse = 1.5647, valid_rmse_diff = 1.4e-05
# seed = 1048
#     only biases: results: counter =  19, max_norm_diff = 0.04, valid_rmse = 1.5603, valid_rmse_diff = 7.9e-06
#     k =   1,     results: counter =  21, max_norm_diff = 0.12, valid_rmse = 1.5600, valid_rmse_diff = 9.6e-06
#     k =   2,     results: counter =  18, max_norm_diff = 0.16, valid_rmse = 1.5599, valid_rmse_diff = 9.8e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5599, valid_rmse_diff = 1e-05
#     k =   8,     results: counter =  18, max_norm_diff = 0.19, valid_rmse = 1.5595, valid_rmse_diff = 9e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.29, valid_rmse = 1.5597, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.26, valid_rmse = 1.5589, valid_rmse_diff = 3.4e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.17, valid_rmse = 1.5594, valid_rmse_diff = 8.2e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.74, valid_rmse = 1.5588, valid_rmse_diff = 5.8e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.78, valid_rmse = 1.5581, valid_rmse_diff = 2.4e-06
# seed = 1049
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5468, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.29, valid_rmse = 1.5464, valid_rmse_diff = 7.2e-06
#     k =   2,     results: counter =   9, max_norm_diff = 0.72, valid_rmse = 1.5469, valid_rmse_diff = 8.8e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.17, valid_rmse = 1.5465, valid_rmse_diff = 7.6e-06
#     k =   8,     results: counter =  21, max_norm_diff = 0.18, valid_rmse = 1.5465, valid_rmse_diff = 1e-05
#     k =  16,     results: counter =  15, max_norm_diff = 0.34, valid_rmse = 1.5462, valid_rmse_diff = 9.3e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.99, valid_rmse = 1.5462, valid_rmse_diff = 3.3e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.27, valid_rmse = 1.5463, valid_rmse_diff = 7.6e-06
#     k = 128,     results: counter =  16, max_norm_diff = 1.21, valid_rmse = 1.5464, valid_rmse_diff = 6.1e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.78, valid_rmse = 1.5455, valid_rmse_diff = 1.3e-05
# seed = 1050
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5784, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  17, max_norm_diff = 0.14, valid_rmse = 1.5782, valid_rmse_diff = 8.5e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.37, valid_rmse = 1.5783, valid_rmse_diff = 6.2e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5781, valid_rmse_diff = 9.1e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.22, valid_rmse = 1.5781, valid_rmse_diff = 7.2e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.32, valid_rmse = 1.5781, valid_rmse_diff = 6.8e-06
#     k =  32,     results: counter =  17, max_norm_diff = 0.80, valid_rmse = 1.5775, valid_rmse_diff = 6.3e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.04, valid_rmse = 1.5776, valid_rmse_diff = 3.9e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.56, valid_rmse = 1.5771, valid_rmse_diff = 4.7e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.76, valid_rmse = 1.5767, valid_rmse_diff = 4.9e-06
# seed = 1051
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5593, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =   6, max_norm_diff = 0.94, valid_rmse = 1.5600, valid_rmse_diff = 0.00012
#     k =   2,     results: counter =  13, max_norm_diff = 0.37, valid_rmse = 1.5594, valid_rmse_diff = 8.9e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5589, valid_rmse_diff = 9e-06
#     k =   8,     results: counter =  24, max_norm_diff = 0.16, valid_rmse = 1.5587, valid_rmse_diff = 9.6e-06
#     k =  16,     results: counter =  21, max_norm_diff = 0.23, valid_rmse = 1.5584, valid_rmse_diff = 9.4e-06
#     k =  32,     results: counter =  24, max_norm_diff = 0.55, valid_rmse = 1.5581, valid_rmse_diff = 9.3e-06
#     k =  64,     results: counter =  22, max_norm_diff = 0.63, valid_rmse = 1.5580, valid_rmse_diff = 8.7e-06
#     k = 128,     results: counter =  19, max_norm_diff = 0.96, valid_rmse = 1.5573, valid_rmse_diff = 9.4e-06
#     k = 256,     results: counter =  14, max_norm_diff = 1.59, valid_rmse = 1.5571, valid_rmse_diff = 8.8e-06
# seed = 1052
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5575, valid_rmse_diff = 8e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.29, valid_rmse = 1.5571, valid_rmse_diff = 7.3e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5569, valid_rmse_diff = 8.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5571, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.25, valid_rmse = 1.5569, valid_rmse_diff = 9e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.42, valid_rmse = 1.5566, valid_rmse_diff = 2.1e-06
#     k =  32,     results: counter =  31, max_norm_diff = 0.40, valid_rmse = 1.5562, valid_rmse_diff = 9.5e-06
#     k =  64,     results: counter =  16, max_norm_diff = 0.99, valid_rmse = 1.5555, valid_rmse_diff = 7.2e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.45, valid_rmse = 1.5546, valid_rmse_diff = 6.9e-06
#     k = 256,     results: counter =  16, max_norm_diff = 1.35, valid_rmse = 1.5546, valid_rmse_diff = 7.7e-06
# seed = 1053
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5495, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  21, max_norm_diff = 0.25, valid_rmse = 1.5491, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5491, valid_rmse_diff = 7.2e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.08, valid_rmse = 1.5492, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5489, valid_rmse_diff = 9.5e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.24, valid_rmse = 1.5488, valid_rmse_diff = 8.8e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.96, valid_rmse = 1.5476, valid_rmse_diff = 6.2e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.82, valid_rmse = 1.5481, valid_rmse_diff = 5.2e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.32, valid_rmse = 1.5477, valid_rmse_diff = 8.4e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.17, valid_rmse = 1.5477, valid_rmse_diff = 1.4e-05
# seed = 1054
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5779, valid_rmse_diff = 8.4e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.15, valid_rmse = 1.5778, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.18, valid_rmse = 1.5774, valid_rmse_diff = 1e-05
#     k =   4,     results: counter =  18, max_norm_diff = 0.13, valid_rmse = 1.5774, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  26, max_norm_diff = 0.14, valid_rmse = 1.5774, valid_rmse_diff = 9.9e-06
#     k =  16,     results: counter =  22, max_norm_diff = 0.18, valid_rmse = 1.5770, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  23, max_norm_diff = 0.60, valid_rmse = 1.5763, valid_rmse_diff = 8.4e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.84, valid_rmse = 1.5764, valid_rmse_diff = 9.4e-06
#     k = 128,     results: counter =   7, max_norm_diff = 3.36, valid_rmse = 1.5757, valid_rmse_diff = 2e-05
#     k = 256,     results: counter =   6, max_norm_diff = 4.57, valid_rmse = 1.5755, valid_rmse_diff = 2.7e-05
# seed = 1055
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5601, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.09, valid_rmse = 1.5598, valid_rmse_diff = 8.6e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.16, valid_rmse = 1.5601, valid_rmse_diff = 9.8e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5598, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.18, valid_rmse = 1.5598, valid_rmse_diff = 8.1e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.30, valid_rmse = 1.5598, valid_rmse_diff = 7.3e-06
#     k =  32,     results: counter =  17, max_norm_diff = 0.81, valid_rmse = 1.5591, valid_rmse_diff = 9e-06
#     k =  64,     results: counter =  11, max_norm_diff = 1.54, valid_rmse = 1.5580, valid_rmse_diff = 1.7e-05
#     k = 128,     results: counter =  15, max_norm_diff = 1.32, valid_rmse = 1.5592, valid_rmse_diff = 5.5e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.20, valid_rmse = 1.5580, valid_rmse_diff = 3.3e-05
# seed = 1056
#     only biases: results: counter =  13, max_norm_diff = 0.16, valid_rmse = 1.5562, valid_rmse_diff = 7.6e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5560, valid_rmse_diff = 8.5e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.16, valid_rmse = 1.5558, valid_rmse_diff = 9.7e-06
#     k =   4,     results: counter =  13, max_norm_diff = 0.18, valid_rmse = 1.5560, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.18, valid_rmse = 1.5558, valid_rmse_diff = 9.6e-06
#     k =  16,     results: counter =  13, max_norm_diff = 0.40, valid_rmse = 1.5554, valid_rmse_diff = 7.1e-06
#     k =  32,     results: counter =  29, max_norm_diff = 0.44, valid_rmse = 1.5544, valid_rmse_diff = 9.8e-06
#     k =  64,     results: counter =  33, max_norm_diff = 0.36, valid_rmse = 1.5543, valid_rmse_diff = 9.1e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.33, valid_rmse = 1.5539, valid_rmse_diff = 1e-05
#     k = 256,     results: counter =  10, max_norm_diff = 2.45, valid_rmse = 1.5527, valid_rmse_diff = 1.1e-05
# seed = 1057
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5745, valid_rmse_diff = 8.3e-06
#     k =   1,     results: counter =  13, max_norm_diff = 0.32, valid_rmse = 1.5746, valid_rmse_diff = 7.8e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.15, valid_rmse = 1.5745, valid_rmse_diff = 9.7e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5743, valid_rmse_diff = 9.2e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.22, valid_rmse = 1.5742, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.32, valid_rmse = 1.5741, valid_rmse_diff = 8.9e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.23, valid_rmse = 1.5742, valid_rmse_diff = 8.3e-07
#     k =  64,     results: counter =  13, max_norm_diff = 1.26, valid_rmse = 1.5738, valid_rmse_diff = 6.4e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.74, valid_rmse = 1.5732, valid_rmse_diff = 1.3e-05
#     k = 256,     results: counter =   8, max_norm_diff = 3.20, valid_rmse = 1.5726, valid_rmse_diff = 1.8e-05
# seed = 1058
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5431, valid_rmse_diff = 7.5e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.26, valid_rmse = 1.5429, valid_rmse_diff = 5.7e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.27, valid_rmse = 1.5429, valid_rmse_diff = 5.7e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.14, valid_rmse = 1.5431, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.21, valid_rmse = 1.5429, valid_rmse_diff = 7.7e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.29, valid_rmse = 1.5432, valid_rmse_diff = 9.8e-06
#     k =  32,     results: counter =  22, max_norm_diff = 0.62, valid_rmse = 1.5420, valid_rmse_diff = 7.4e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.05, valid_rmse = 1.5424, valid_rmse_diff = 9.8e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5412, valid_rmse_diff = 1.4e-05
#     k = 256,     results: counter =  11, max_norm_diff = 2.18, valid_rmse = 1.5415, valid_rmse_diff = 7.2e-08
# seed = 1059
#     only biases: results: counter =  19, max_norm_diff = 0.04, valid_rmse = 1.5726, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =  10, max_norm_diff = 0.62, valid_rmse = 1.5732, valid_rmse_diff = 3.9e-06
#     k =   2,     results: counter =  13, max_norm_diff = 0.39, valid_rmse = 1.5731, valid_rmse_diff = 8.2e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.12, valid_rmse = 1.5725, valid_rmse_diff = 8.3e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.16, valid_rmse = 1.5722, valid_rmse_diff = 8.9e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.25, valid_rmse = 1.5722, valid_rmse_diff = 8.8e-06
#     k =  32,     results: counter =  17, max_norm_diff = 0.83, valid_rmse = 1.5720, valid_rmse_diff = 6.4e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.16, valid_rmse = 1.5716, valid_rmse_diff = 3.8e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.46, valid_rmse = 1.5720, valid_rmse_diff = 4e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.22, valid_rmse = 1.5714, valid_rmse_diff = 1e-05
# seed = 1060
#     only biases: results: counter =  11, max_norm_diff = 0.26, valid_rmse = 1.5648, valid_rmse_diff = 9.9e-06
#     k =   1,     results: counter =  11, max_norm_diff = 0.41, valid_rmse = 1.5645, valid_rmse_diff = 1.2e-06
#     k =   2,     results: counter =   6, max_norm_diff = 1.22, valid_rmse = 1.5648, valid_rmse_diff = 4.1e-05
#     k =   4,     results: counter =  12, max_norm_diff = 0.21, valid_rmse = 1.5645, valid_rmse_diff = 8.1e-06
#     k =   8,     results: counter =  11, max_norm_diff = 0.29, valid_rmse = 1.5644, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  10, max_norm_diff = 0.49, valid_rmse = 1.5642, valid_rmse_diff = 7.4e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.24, valid_rmse = 1.5634, valid_rmse_diff = 9.4e-06
#     k =  64,     results: counter =  10, max_norm_diff = 1.70, valid_rmse = 1.5635, valid_rmse_diff = 5.7e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.17, valid_rmse = 1.5625, valid_rmse_diff = 9.2e-06
#     k = 256,     results: counter =   6, max_norm_diff = 4.57, valid_rmse = 1.5625, valid_rmse_diff = 4.4e-05
# seed = 1061
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5652, valid_rmse_diff = 8.3e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5652, valid_rmse_diff = 7.2e-06
#     k =   2,     results: counter =  13, max_norm_diff = 0.38, valid_rmse = 1.5647, valid_rmse_diff = 1e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.13, valid_rmse = 1.5648, valid_rmse_diff = 9.8e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.23, valid_rmse = 1.5647, valid_rmse_diff = 9.3e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.36, valid_rmse = 1.5646, valid_rmse_diff = 6.2e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.38, valid_rmse = 1.5641, valid_rmse_diff = 9.3e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.28, valid_rmse = 1.5639, valid_rmse_diff = 3.5e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.59, valid_rmse = 1.5637, valid_rmse_diff = 1.4e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.19, valid_rmse = 1.5633, valid_rmse_diff = 4.9e-07
# seed = 1062
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5648, valid_rmse_diff = 7.7e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.16, valid_rmse = 1.5646, valid_rmse_diff = 8.8e-06
#     k =   2,     results: counter =  21, max_norm_diff = 0.14, valid_rmse = 1.5643, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5645, valid_rmse_diff = 8.1e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.19, valid_rmse = 1.5645, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.37, valid_rmse = 1.5643, valid_rmse_diff = 8.5e-06
#     k =  32,     results: counter =  10, max_norm_diff = 1.45, valid_rmse = 1.5644, valid_rmse_diff = 1.7e-05
#     k =  64,     results: counter =  10, max_norm_diff = 1.70, valid_rmse = 1.5642, valid_rmse_diff = 8.9e-07
#     k = 128,     results: counter =   9, max_norm_diff = 2.43, valid_rmse = 1.5641, valid_rmse_diff = 6e-06
#     k = 256,     results: counter =   6, max_norm_diff = 4.56, valid_rmse = 1.5629, valid_rmse_diff = 2.3e-05
# seed = 1063
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5655, valid_rmse_diff = 9.8e-06
#     k =   1,     results: counter =   5, max_norm_diff = 1.10, valid_rmse = 1.5658, valid_rmse_diff = 9.1e-05
#     k =   2,     results: counter =   4, max_norm_diff = 2.03, valid_rmse = 1.5661, valid_rmse_diff = 0.00013
#     k =   4,     results: counter =  19, max_norm_diff = 0.13, valid_rmse = 1.5649, valid_rmse_diff = 8.5e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.25, valid_rmse = 1.5649, valid_rmse_diff = 8.7e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.30, valid_rmse = 1.5647, valid_rmse_diff = 7.2e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.20, valid_rmse = 1.5642, valid_rmse_diff = 6.3e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.25, valid_rmse = 1.5638, valid_rmse_diff = 6.9e-06
#     k = 128,     results: counter =  16, max_norm_diff = 1.21, valid_rmse = 1.5635, valid_rmse_diff = 9.1e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.76, valid_rmse = 1.5620, valid_rmse_diff = 1.4e-05
# seed = 1064
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5613, valid_rmse_diff = 7.8e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.12, valid_rmse = 1.5613, valid_rmse_diff = 8.6e-06
#     k =   2,     results: counter =  24, max_norm_diff = 0.14, valid_rmse = 1.5612, valid_rmse_diff = 9.9e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5611, valid_rmse_diff = 9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.20, valid_rmse = 1.5611, valid_rmse_diff = 9.5e-06
#     k =  16,     results: counter =  11, max_norm_diff = 0.43, valid_rmse = 1.5611, valid_rmse_diff = 2.6e-06
#     k =  32,     results: counter =   9, max_norm_diff = 1.69, valid_rmse = 1.5605, valid_rmse_diff = 1.9e-05
#     k =  64,     results: counter =  12, max_norm_diff = 1.37, valid_rmse = 1.5604, valid_rmse_diff = 3.9e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5599, valid_rmse_diff = 9.9e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5597, valid_rmse_diff = 2.3e-05
# seed = 1065
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5490, valid_rmse_diff = 7.7e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.25, valid_rmse = 1.5487, valid_rmse_diff = 7.7e-06
#     k =   2,     results: counter =  10, max_norm_diff = 0.44, valid_rmse = 1.5489, valid_rmse_diff = 9.5e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.17, valid_rmse = 1.5488, valid_rmse_diff = 8.1e-06
#     k =   8,     results: counter =  25, max_norm_diff = 0.14, valid_rmse = 1.5485, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.33, valid_rmse = 1.5488, valid_rmse_diff = 9e-06
#     k =  32,     results: counter =   9, max_norm_diff = 1.75, valid_rmse = 1.5483, valid_rmse_diff = 2.4e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.05, valid_rmse = 1.5482, valid_rmse_diff = 4.6e-06
#     k = 128,     results: counter =   8, max_norm_diff = 2.85, valid_rmse = 1.5477, valid_rmse_diff = 1.5e-05
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5474, valid_rmse_diff = 1.2e-05
# seed = 1066
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5701, valid_rmse_diff = 8e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.09, valid_rmse = 1.5700, valid_rmse_diff = 9.9e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5701, valid_rmse_diff = 9.9e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.13, valid_rmse = 1.5699, valid_rmse_diff = 7.9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.22, valid_rmse = 1.5698, valid_rmse_diff = 8.7e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.28, valid_rmse = 1.5699, valid_rmse_diff = 8.4e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.24, valid_rmse = 1.5695, valid_rmse_diff = 1.8e-08
#     k =  64,     results: counter =  15, max_norm_diff = 1.06, valid_rmse = 1.5700, valid_rmse_diff = 2.8e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.48, valid_rmse = 1.5704, valid_rmse_diff = 1.2e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.78, valid_rmse = 1.5702, valid_rmse_diff = 2.2e-05
# seed = 1067
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5711, valid_rmse_diff = 8e-06
#     k =   1,     results: counter =   8, max_norm_diff = 0.59, valid_rmse = 1.5711, valid_rmse_diff = 3.8e-05
#     k =   2,     results: counter =  13, max_norm_diff = 0.24, valid_rmse = 1.5707, valid_rmse_diff = 4.3e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5707, valid_rmse_diff = 9.8e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.16, valid_rmse = 1.5704, valid_rmse_diff = 8.6e-06
#     k =  16,     results: counter =  19, max_norm_diff = 0.24, valid_rmse = 1.5703, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  27, max_norm_diff = 0.47, valid_rmse = 1.5686, valid_rmse_diff = 9.7e-06
#     k =  64,     results: counter =  21, max_norm_diff = 0.69, valid_rmse = 1.5685, valid_rmse_diff = 9.9e-06
#     k = 128,     results: counter =  26, max_norm_diff = 0.60, valid_rmse = 1.5684, valid_rmse_diff = 9.4e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.95, valid_rmse = 1.5678, valid_rmse_diff = 8.1e-06
# seed = 1068
#     only biases: results: counter =  12, max_norm_diff = 0.20, valid_rmse = 1.5598, valid_rmse_diff = 7.4e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.15, valid_rmse = 1.5595, valid_rmse_diff = 7.1e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5595, valid_rmse_diff = 9.9e-06
#     k =   4,     results: counter =  12, max_norm_diff = 0.21, valid_rmse = 1.5596, valid_rmse_diff = 6.9e-06
#     k =   8,     results: counter =  11, max_norm_diff = 0.34, valid_rmse = 1.5596, valid_rmse_diff = 5.6e-06
#     k =  16,     results: counter =  10, max_norm_diff = 0.51, valid_rmse = 1.5594, valid_rmse_diff = 7.4e-06
#     k =  32,     results: counter =  18, max_norm_diff = 0.78, valid_rmse = 1.5587, valid_rmse_diff = 7.3e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.14, valid_rmse = 1.5583, valid_rmse_diff = 9.3e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.32, valid_rmse = 1.5576, valid_rmse_diff = 7.5e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5571, valid_rmse_diff = 9.1e-06
# seed = 1069
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5486, valid_rmse_diff = 7.8e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5482, valid_rmse_diff = 8.4e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5483, valid_rmse_diff = 8.3e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.13, valid_rmse = 1.5483, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.19, valid_rmse = 1.5480, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.30, valid_rmse = 1.5480, valid_rmse_diff = 8.9e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.06, valid_rmse = 1.5477, valid_rmse_diff = 2.8e-06
#     k =  64,     results: counter =  19, max_norm_diff = 0.78, valid_rmse = 1.5480, valid_rmse_diff = 6.1e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.47, valid_rmse = 1.5475, valid_rmse_diff = 9.8e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.78, valid_rmse = 1.5476, valid_rmse_diff = 2.7e-05
# seed = 1070
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5596, valid_rmse_diff = 9.9e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5594, valid_rmse_diff = 9.1e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.21, valid_rmse = 1.5595, valid_rmse_diff = 9.1e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.14, valid_rmse = 1.5593, valid_rmse_diff = 9.6e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.23, valid_rmse = 1.5593, valid_rmse_diff = 9.2e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.44, valid_rmse = 1.5589, valid_rmse_diff = 2.3e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.97, valid_rmse = 1.5596, valid_rmse_diff = 8.5e-06
#     k =  64,     results: counter =  12, max_norm_diff = 1.37, valid_rmse = 1.5589, valid_rmse_diff = 8e-07
#     k = 128,     results: counter =   9, max_norm_diff = 2.50, valid_rmse = 1.5593, valid_rmse_diff = 3e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.80, valid_rmse = 1.5584, valid_rmse_diff = 2.9e-05
# seed = 1071
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5609, valid_rmse_diff = 9.6e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.17, valid_rmse = 1.5604, valid_rmse_diff = 9.5e-06
#     k =   2,     results: counter =  26, max_norm_diff = 0.19, valid_rmse = 1.5604, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5605, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.17, valid_rmse = 1.5604, valid_rmse_diff = 8.6e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.30, valid_rmse = 1.5601, valid_rmse_diff = 9.6e-06
#     k =  32,     results: counter =  10, max_norm_diff = 1.53, valid_rmse = 1.5601, valid_rmse_diff = 2.3e-05
#     k =  64,     results: counter =  11, max_norm_diff = 1.54, valid_rmse = 1.5599, valid_rmse_diff = 1.8e-07
#     k = 128,     results: counter =   9, max_norm_diff = 2.47, valid_rmse = 1.5597, valid_rmse_diff = 2.1e-05
#     k = 256,     results: counter =   7, max_norm_diff = 3.80, valid_rmse = 1.5589, valid_rmse_diff = 7.6e-05
# seed = 1072
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5719, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.09, valid_rmse = 1.5714, valid_rmse_diff = 8e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.19, valid_rmse = 1.5713, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.14, valid_rmse = 1.5716, valid_rmse_diff = 9.9e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.17, valid_rmse = 1.5715, valid_rmse_diff = 8.9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.25, valid_rmse = 1.5712, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  36, max_norm_diff = 0.34, valid_rmse = 1.5702, valid_rmse_diff = 9.1e-06
#     k =  64,     results: counter =  23, max_norm_diff = 0.61, valid_rmse = 1.5701, valid_rmse_diff = 8.9e-06
#     k = 128,     results: counter =  17, max_norm_diff = 1.11, valid_rmse = 1.5696, valid_rmse_diff = 6.6e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.79, valid_rmse = 1.5688, valid_rmse_diff = 1.9e-05
# seed = 1073
#     only biases: results: counter =  15, max_norm_diff = 0.10, valid_rmse = 1.5536, valid_rmse_diff = 8.2e-06
#     k =   1,     results: counter =  10, max_norm_diff = 0.42, valid_rmse = 1.5533, valid_rmse_diff = 3.4e-06
#     k =   2,     results: counter =   6, max_norm_diff = 1.15, valid_rmse = 1.5540, valid_rmse_diff = 4.1e-05
#     k =   4,     results: counter =  12, max_norm_diff = 0.21, valid_rmse = 1.5535, valid_rmse_diff = 9.2e-06
#     k =   8,     results: counter =  22, max_norm_diff = 0.16, valid_rmse = 1.5533, valid_rmse_diff = 9.6e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.31, valid_rmse = 1.5530, valid_rmse_diff = 6.6e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.90, valid_rmse = 1.5540, valid_rmse_diff = 9.6e-06
#     k =  64,     results: counter =  21, max_norm_diff = 0.68, valid_rmse = 1.5538, valid_rmse_diff = 9.4e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.18, valid_rmse = 1.5533, valid_rmse_diff = 6.5e-06
#     k = 256,     results: counter =   6, max_norm_diff = 4.55, valid_rmse = 1.5533, valid_rmse_diff = 1e-05
# seed = 1074
#     only biases: results: counter =  13, max_norm_diff = 0.16, valid_rmse = 1.5650, valid_rmse_diff = 8.8e-06
#     k =   1,     results: counter =  11, max_norm_diff = 0.41, valid_rmse = 1.5646, valid_rmse_diff = 1.1e-05
#     k =   2,     results: counter =  12, max_norm_diff = 0.24, valid_rmse = 1.5647, valid_rmse_diff = 9.4e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.14, valid_rmse = 1.5645, valid_rmse_diff = 9.7e-06
#     k =   8,     results: counter =  14, max_norm_diff = 0.23, valid_rmse = 1.5641, valid_rmse_diff = 7.1e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.29, valid_rmse = 1.5642, valid_rmse_diff = 8.4e-06
#     k =  32,     results: counter =   9, max_norm_diff = 1.71, valid_rmse = 1.5632, valid_rmse_diff = 1.2e-05
#     k =  64,     results: counter =  27, max_norm_diff = 0.48, valid_rmse = 1.5643, valid_rmse_diff = 9.1e-06
#     k = 128,     results: counter =  11, max_norm_diff = 1.95, valid_rmse = 1.5633, valid_rmse_diff = 7.6e-07
#     k = 256,     results: counter =   7, max_norm_diff = 3.78, valid_rmse = 1.5625, valid_rmse_diff = 4.2e-05
# seed = 1075
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5525, valid_rmse_diff = 8.5e-06
#     k =   1,     results: counter =  17, max_norm_diff = 0.16, valid_rmse = 1.5528, valid_rmse_diff = 8.2e-06
#     k =   2,     results: counter =  12, max_norm_diff = 0.36, valid_rmse = 1.5527, valid_rmse_diff = 2.5e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.14, valid_rmse = 1.5524, valid_rmse_diff = 9e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5524, valid_rmse_diff = 1e-05
#     k =  16,     results: counter =  16, max_norm_diff = 0.31, valid_rmse = 1.5521, valid_rmse_diff = 8.8e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.96, valid_rmse = 1.5512, valid_rmse_diff = 8.9e-07
#     k =  64,     results: counter =  15, max_norm_diff = 1.07, valid_rmse = 1.5519, valid_rmse_diff = 9.5e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.43, valid_rmse = 1.5520, valid_rmse_diff = 7.4e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5513, valid_rmse_diff = 1.2e-05
# seed = 1076
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5631, valid_rmse_diff = 8.1e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.20, valid_rmse = 1.5628, valid_rmse_diff = 5.2e-06
#     k =   2,     results: counter =  23, max_norm_diff = 0.15, valid_rmse = 1.5632, valid_rmse_diff = 9.3e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.11, valid_rmse = 1.5628, valid_rmse_diff = 8.7e-06
#     k =   8,     results: counter =  13, max_norm_diff = 0.23, valid_rmse = 1.5628, valid_rmse_diff = 6.1e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.32, valid_rmse = 1.5629, valid_rmse_diff = 6.6e-06
#     k =  32,     results: counter =  18, max_norm_diff = 0.79, valid_rmse = 1.5630, valid_rmse_diff = 5.8e-06
#     k =  64,     results: counter =  11, max_norm_diff = 1.53, valid_rmse = 1.5622, valid_rmse_diff = 2e-05
#     k = 128,     results: counter =  10, max_norm_diff = 2.18, valid_rmse = 1.5621, valid_rmse_diff = 4.9e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.77, valid_rmse = 1.5615, valid_rmse_diff = 5.4e-06
# seed = 1077
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5578, valid_rmse_diff = 9.3e-06
#     k =   1,     results: counter =  17, max_norm_diff = 0.13, valid_rmse = 1.5571, valid_rmse_diff = 7.2e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5573, valid_rmse_diff = 9.2e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.11, valid_rmse = 1.5573, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  21, max_norm_diff = 0.17, valid_rmse = 1.5570, valid_rmse_diff = 8.8e-06
#     k =  16,     results: counter =  22, max_norm_diff = 0.21, valid_rmse = 1.5568, valid_rmse_diff = 8.8e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.09, valid_rmse = 1.5565, valid_rmse_diff = 8.8e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.28, valid_rmse = 1.5559, valid_rmse_diff = 4.4e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5555, valid_rmse_diff = 4.9e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5554, valid_rmse_diff = 2.6e-05
# seed = 1078
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5527, valid_rmse_diff = 7.9e-06
#     k =   1,     results: counter =  17, max_norm_diff = 0.08, valid_rmse = 1.5526, valid_rmse_diff = 7.3e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.18, valid_rmse = 1.5524, valid_rmse_diff = 8.8e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5525, valid_rmse_diff = 7.9e-06
#     k =   8,     results: counter =  12, max_norm_diff = 0.30, valid_rmse = 1.5524, valid_rmse_diff = 7.1e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5522, valid_rmse_diff = 8.6e-06
#     k =  32,     results: counter =  15, max_norm_diff = 0.97, valid_rmse = 1.5515, valid_rmse_diff = 8.5e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.83, valid_rmse = 1.5514, valid_rmse_diff = 9e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5504, valid_rmse_diff = 6e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.79, valid_rmse = 1.5501, valid_rmse_diff = 2.5e-06
# seed = 1079
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5638, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.04, valid_rmse = 1.5634, valid_rmse_diff = 7.7e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.18, valid_rmse = 1.5631, valid_rmse_diff = 8.2e-06
#     k =   4,     results: counter =  19, max_norm_diff = 0.12, valid_rmse = 1.5633, valid_rmse_diff = 8.6e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.22, valid_rmse = 1.5631, valid_rmse_diff = 7.8e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.32, valid_rmse = 1.5632, valid_rmse_diff = 7.7e-06
#     k =  32,     results: counter =  28, max_norm_diff = 0.46, valid_rmse = 1.5620, valid_rmse_diff = 8.6e-06
#     k =  64,     results: counter =  17, max_norm_diff = 0.91, valid_rmse = 1.5622, valid_rmse_diff = 7.4e-06
#     k = 128,     results: counter =  18, max_norm_diff = 1.03, valid_rmse = 1.5622, valid_rmse_diff = 9.4e-06
#     k = 256,     results: counter =  16, max_norm_diff = 1.33, valid_rmse = 1.5623, valid_rmse_diff = 7.9e-06
# seed = 1080
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5428, valid_rmse_diff = 9.8e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.25, valid_rmse = 1.5425, valid_rmse_diff = 7.2e-06
#     k =   2,     results: counter =   9, max_norm_diff = 0.50, valid_rmse = 1.5431, valid_rmse_diff = 1.8e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.12, valid_rmse = 1.5426, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.19, valid_rmse = 1.5423, valid_rmse_diff = 6.9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.25, valid_rmse = 1.5422, valid_rmse_diff = 9.7e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.06, valid_rmse = 1.5403, valid_rmse_diff = 7.7e-06
#     k =  64,     results: counter =  22, max_norm_diff = 0.63, valid_rmse = 1.5404, valid_rmse_diff = 9.8e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.33, valid_rmse = 1.5400, valid_rmse_diff = 9.6e-06
#     k = 256,     results: counter =  12, max_norm_diff = 1.94, valid_rmse = 1.5397, valid_rmse_diff = 1.6e-06
# seed = 1081
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5736, valid_rmse_diff = 8.7e-06
#     k =   1,     results: counter =  20, max_norm_diff = 0.04, valid_rmse = 1.5730, valid_rmse_diff = 8e-06
#     k =   2,     results: counter =   6, max_norm_diff = 1.20, valid_rmse = 1.5741, valid_rmse_diff = 9.3e-05
#     k =   4,     results: counter =  18, max_norm_diff = 0.10, valid_rmse = 1.5731, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.15, valid_rmse = 1.5729, valid_rmse_diff = 7.8e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.29, valid_rmse = 1.5728, valid_rmse_diff = 8e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.39, valid_rmse = 1.5730, valid_rmse_diff = 7.6e-06
#     k =  64,     results: counter =  10, max_norm_diff = 1.70, valid_rmse = 1.5729, valid_rmse_diff = 6.6e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.46, valid_rmse = 1.5727, valid_rmse_diff = 1.1e-05
#     k = 256,     results: counter =   8, max_norm_diff = 3.17, valid_rmse = 1.5728, valid_rmse_diff = 1.2e-05
# seed = 1082
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5574, valid_rmse_diff = 7.3e-06
#     k =   1,     results: counter =  13, max_norm_diff = 0.25, valid_rmse = 1.5573, valid_rmse_diff = 2.6e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5570, valid_rmse_diff = 9.8e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.18, valid_rmse = 1.5571, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.20, valid_rmse = 1.5570, valid_rmse_diff = 8.5e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.26, valid_rmse = 1.5568, valid_rmse_diff = 9.1e-06
#     k =  32,     results: counter =  28, max_norm_diff = 0.46, valid_rmse = 1.5565, valid_rmse_diff = 9.7e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.16, valid_rmse = 1.5559, valid_rmse_diff = 9.5e-06
#     k = 128,     results: counter =  17, max_norm_diff = 1.13, valid_rmse = 1.5562, valid_rmse_diff = 8.5e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.18, valid_rmse = 1.5559, valid_rmse_diff = 1.2e-05
# seed = 1083
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5582, valid_rmse_diff = 9.5e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.24, valid_rmse = 1.5580, valid_rmse_diff = 8.9e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.26, valid_rmse = 1.5579, valid_rmse_diff = 7.6e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5579, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  24, max_norm_diff = 0.14, valid_rmse = 1.5578, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  18, max_norm_diff = 0.24, valid_rmse = 1.5575, valid_rmse_diff = 8.5e-06
#     k =  32,     results: counter =  19, max_norm_diff = 0.73, valid_rmse = 1.5570, valid_rmse_diff = 7.3e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.16, valid_rmse = 1.5562, valid_rmse_diff = 3.7e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.57, valid_rmse = 1.5558, valid_rmse_diff = 1.1e-05
#     k = 256,     results: counter =   9, max_norm_diff = 2.77, valid_rmse = 1.5549, valid_rmse_diff = 4.5e-06
# seed = 1084
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5541, valid_rmse_diff = 9.7e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.16, valid_rmse = 1.5536, valid_rmse_diff = 8.9e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.40, valid_rmse = 1.5537, valid_rmse_diff = 6.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.14, valid_rmse = 1.5536, valid_rmse_diff = 9.2e-06
#     k =   8,     results: counter =  19, max_norm_diff = 0.16, valid_rmse = 1.5535, valid_rmse_diff = 9.1e-06
#     k =  16,     results: counter =  20, max_norm_diff = 0.23, valid_rmse = 1.5533, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  21, max_norm_diff = 0.65, valid_rmse = 1.5531, valid_rmse_diff = 1e-05
#     k =  64,     results: counter =  16, max_norm_diff = 0.96, valid_rmse = 1.5528, valid_rmse_diff = 3.8e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.59, valid_rmse = 1.5521, valid_rmse_diff = 1.9e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.45, valid_rmse = 1.5526, valid_rmse_diff = 1.6e-05
# seed = 1085
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5566, valid_rmse_diff = 7.6e-06
#     k =   1,     results: counter =  18, max_norm_diff = 0.14, valid_rmse = 1.5564, valid_rmse_diff = 9.7e-06
#     k =   2,     results: counter =  27, max_norm_diff = 0.16, valid_rmse = 1.5560, valid_rmse_diff = 9.6e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.16, valid_rmse = 1.5563, valid_rmse_diff = 8.6e-06
#     k =   8,     results: counter =  13, max_norm_diff = 0.27, valid_rmse = 1.5563, valid_rmse_diff = 8.3e-06
#     k =  16,     results: counter =  12, max_norm_diff = 0.39, valid_rmse = 1.5561, valid_rmse_diff = 6.2e-06
#     k =  32,     results: counter =  18, max_norm_diff = 0.81, valid_rmse = 1.5556, valid_rmse_diff = 7.1e-06
#     k =  64,     results: counter =  10, max_norm_diff = 1.72, valid_rmse = 1.5553, valid_rmse_diff = 8.6e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.20, valid_rmse = 1.5555, valid_rmse_diff = 2.9e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.46, valid_rmse = 1.5551, valid_rmse_diff = 7.7e-06
# seed = 1086
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5745, valid_rmse_diff = 9.4e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5743, valid_rmse_diff = 9.5e-06
#     k =   2,     results: counter =  27, max_norm_diff = 0.15, valid_rmse = 1.5739, valid_rmse_diff = 9.9e-06
#     k =   4,     results: counter =  20, max_norm_diff = 0.12, valid_rmse = 1.5741, valid_rmse_diff = 8.8e-06
#     k =   8,     results: counter =  24, max_norm_diff = 0.13, valid_rmse = 1.5738, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  20, max_norm_diff = 0.24, valid_rmse = 1.5736, valid_rmse_diff = 8.1e-06
#     k =  32,     results: counter =  13, max_norm_diff = 1.14, valid_rmse = 1.5735, valid_rmse_diff = 1.5e-06
#     k =  64,     results: counter =  12, max_norm_diff = 1.40, valid_rmse = 1.5736, valid_rmse_diff = 9.3e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.76, valid_rmse = 1.5740, valid_rmse_diff = 7.8e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.78, valid_rmse = 1.5732, valid_rmse_diff = 2.6e-06
# seed = 1087
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5673, valid_rmse_diff = 9.9e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.27, valid_rmse = 1.5670, valid_rmse_diff = 9e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.18, valid_rmse = 1.5669, valid_rmse_diff = 5.5e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.14, valid_rmse = 1.5668, valid_rmse_diff = 9.3e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.16, valid_rmse = 1.5667, valid_rmse_diff = 8.8e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5664, valid_rmse_diff = 8e-06
#     k =  32,     results: counter =  20, max_norm_diff = 0.71, valid_rmse = 1.5656, valid_rmse_diff = 9.7e-06
#     k =  64,     results: counter =  24, max_norm_diff = 0.57, valid_rmse = 1.5651, valid_rmse_diff = 7.8e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.61, valid_rmse = 1.5647, valid_rmse_diff = 3.1e-06
#     k = 256,     results: counter =  13, max_norm_diff = 1.77, valid_rmse = 1.5648, valid_rmse_diff = 4.8e-06
# seed = 1088
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5621, valid_rmse_diff = 8.7e-06
#     k =   1,     results: counter =  17, max_norm_diff = 0.13, valid_rmse = 1.5619, valid_rmse_diff = 9.6e-06
#     k =   2,     results: counter =   8, max_norm_diff = 0.90, valid_rmse = 1.5622, valid_rmse_diff = 1.5e-05
#     k =   4,     results: counter =  17, max_norm_diff = 0.10, valid_rmse = 1.5617, valid_rmse_diff = 8.5e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.20, valid_rmse = 1.5616, valid_rmse_diff = 7.9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.26, valid_rmse = 1.5615, valid_rmse_diff = 7.7e-06
#     k =  32,     results: counter =  30, max_norm_diff = 0.43, valid_rmse = 1.5608, valid_rmse_diff = 9.3e-06
#     k =  64,     results: counter =  16, max_norm_diff = 1.00, valid_rmse = 1.5603, valid_rmse_diff = 9.3e-06
#     k = 128,     results: counter =  15, max_norm_diff = 1.34, valid_rmse = 1.5602, valid_rmse_diff = 8.5e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.18, valid_rmse = 1.5599, valid_rmse_diff = 8.5e-06
# seed = 1089
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5650, valid_rmse_diff = 8.6e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.13, valid_rmse = 1.5647, valid_rmse_diff = 6.8e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.22, valid_rmse = 1.5646, valid_rmse_diff = 8.1e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.10, valid_rmse = 1.5645, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.21, valid_rmse = 1.5643, valid_rmse_diff = 9.7e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.32, valid_rmse = 1.5644, valid_rmse_diff = 7.2e-06
#     k =  32,     results: counter =  14, max_norm_diff = 1.08, valid_rmse = 1.5636, valid_rmse_diff = 4.2e-06
#     k =  64,     results: counter =  13, max_norm_diff = 1.26, valid_rmse = 1.5640, valid_rmse_diff = 4.4e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5629, valid_rmse_diff = 8.6e-06
#     k = 256,     results: counter =  11, max_norm_diff = 2.19, valid_rmse = 1.5630, valid_rmse_diff = 6.1e-06
# seed = 1090
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5645, valid_rmse_diff = 8.2e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.13, valid_rmse = 1.5641, valid_rmse_diff = 8.6e-06
#     k =   2,     results: counter =  16, max_norm_diff = 0.22, valid_rmse = 1.5643, valid_rmse_diff = 8.1e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.11, valid_rmse = 1.5642, valid_rmse_diff = 8.9e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.22, valid_rmse = 1.5643, valid_rmse_diff = 6.9e-06
#     k =  16,     results: counter =  13, max_norm_diff = 0.35, valid_rmse = 1.5641, valid_rmse_diff = 6.2e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.37, valid_rmse = 1.5632, valid_rmse_diff = 1.4e-05
#     k =  64,     results: counter =  15, max_norm_diff = 1.07, valid_rmse = 1.5634, valid_rmse_diff = 9.7e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.74, valid_rmse = 1.5622, valid_rmse_diff = 2.7e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.77, valid_rmse = 1.5617, valid_rmse_diff = 2.6e-06
# seed = 1091
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5711, valid_rmse_diff = 7.8e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.37, valid_rmse = 1.5706, valid_rmse_diff = 3e-06
#     k =   2,     results: counter =  22, max_norm_diff = 0.19, valid_rmse = 1.5705, valid_rmse_diff = 7.7e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.12, valid_rmse = 1.5706, valid_rmse_diff = 9.2e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.18, valid_rmse = 1.5704, valid_rmse_diff = 9.5e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.31, valid_rmse = 1.5706, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.23, valid_rmse = 1.5695, valid_rmse_diff = 6.3e-06
#     k =  64,     results: counter =  11, max_norm_diff = 1.51, valid_rmse = 1.5702, valid_rmse_diff = 4.6e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5693, valid_rmse_diff = 5.9e-06
#     k = 256,     results: counter =   8, max_norm_diff = 3.21, valid_rmse = 1.5692, valid_rmse_diff = 1.1e-05
# seed = 1092
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5514, valid_rmse_diff = 8.5e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.24, valid_rmse = 1.5510, valid_rmse_diff = 7.9e-06
#     k =   2,     results: counter =  12, max_norm_diff = 0.31, valid_rmse = 1.5513, valid_rmse_diff = 9.9e-06
#     k =   4,     results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5510, valid_rmse_diff = 9.5e-06
#     k =   8,     results: counter =  15, max_norm_diff = 0.26, valid_rmse = 1.5510, valid_rmse_diff = 9.4e-06
#     k =  16,     results: counter =   9, max_norm_diff = 0.59, valid_rmse = 1.5507, valid_rmse_diff = 9e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.37, valid_rmse = 1.5501, valid_rmse_diff = 5.7e-07
#     k =  64,     results: counter =  14, max_norm_diff = 1.14, valid_rmse = 1.5491, valid_rmse_diff = 6.7e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.20, valid_rmse = 1.5482, valid_rmse_diff = 1.2e-05
#     k = 256,     results: counter =   8, max_norm_diff = 3.23, valid_rmse = 1.5471, valid_rmse_diff = 1.1e-05
# seed = 1093
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5678, valid_rmse_diff = 9.7e-06
#     k =   1,     results: counter =  15, max_norm_diff = 0.30, valid_rmse = 1.5673, valid_rmse_diff = 9.6e-06
#     k =   2,     results: counter =  10, max_norm_diff = 0.39, valid_rmse = 1.5678, valid_rmse_diff = 6.4e-06
#     k =   4,     results: counter =  15, max_norm_diff = 0.11, valid_rmse = 1.5674, valid_rmse_diff = 7.7e-06
#     k =   8,     results: counter =  13, max_norm_diff = 0.24, valid_rmse = 1.5675, valid_rmse_diff = 1e-05
#     k =  16,     results: counter =  10, max_norm_diff = 0.52, valid_rmse = 1.5675, valid_rmse_diff = 6.8e-06
#     k =  32,     results: counter =   9, max_norm_diff = 1.69, valid_rmse = 1.5663, valid_rmse_diff = 2.1e-05
#     k =  64,     results: counter =  11, max_norm_diff = 1.56, valid_rmse = 1.5662, valid_rmse_diff = 1.1e-05
#     k = 128,     results: counter =  11, max_norm_diff = 1.95, valid_rmse = 1.5657, valid_rmse_diff = 4.4e-07
#     k = 256,     results: counter =   7, max_norm_diff = 3.79, valid_rmse = 1.5653, valid_rmse_diff = 2.8e-06
# seed = 1094
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5713, valid_rmse_diff = 9e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5715, valid_rmse_diff = 9.4e-06
#     k =   2,     results: counter =  14, max_norm_diff = 0.33, valid_rmse = 1.5718, valid_rmse_diff = 4.6e-06
#     k =   4,     results: counter =  17, max_norm_diff = 0.08, valid_rmse = 1.5712, valid_rmse_diff = 9.4e-06
#     k =   8,     results: counter =  18, max_norm_diff = 0.18, valid_rmse = 1.5712, valid_rmse_diff = 7.9e-06
#     k =  16,     results: counter =  17, max_norm_diff = 0.27, valid_rmse = 1.5708, valid_rmse_diff = 8.9e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.88, valid_rmse = 1.5689, valid_rmse_diff = 7.5e-06
#     k =  64,     results: counter =  27, max_norm_diff = 0.48, valid_rmse = 1.5691, valid_rmse_diff = 8.8e-06
#     k = 128,     results: counter =  14, max_norm_diff = 1.42, valid_rmse = 1.5685, valid_rmse_diff = 5.8e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.78, valid_rmse = 1.5675, valid_rmse_diff = 2.6e-05
# seed = 1095
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5642, valid_rmse_diff = 8.5e-06
#     k =   1,     results: counter =  19, max_norm_diff = 0.09, valid_rmse = 1.5642, valid_rmse_diff = 8e-06
#     k =   2,     results: counter =  20, max_norm_diff = 0.12, valid_rmse = 1.5635, valid_rmse_diff = 1e-05
#     k =   4,     results: counter =  17, max_norm_diff = 0.12, valid_rmse = 1.5638, valid_rmse_diff = 9.8e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.16, valid_rmse = 1.5635, valid_rmse_diff = 8.3e-06
#     k =  16,     results: counter =  16, max_norm_diff = 0.29, valid_rmse = 1.5634, valid_rmse_diff = 9.6e-06
#     k =  32,     results: counter =  12, max_norm_diff = 1.25, valid_rmse = 1.5638, valid_rmse_diff = 2.4e-06
#     k =  64,     results: counter =  10, max_norm_diff = 1.70, valid_rmse = 1.5632, valid_rmse_diff = 6.8e-06
#     k = 128,     results: counter =   9, max_norm_diff = 2.46, valid_rmse = 1.5634, valid_rmse_diff = 5e-06
#     k = 256,     results: counter =   5, max_norm_diff = 5.83, valid_rmse = 1.5632, valid_rmse_diff = 8.3e-05
# seed = 1096
#     only biases: results: counter =  16, max_norm_diff = 0.08, valid_rmse = 1.5516, valid_rmse_diff = 9.1e-06
#     k =   1,     results: counter =  14, max_norm_diff = 0.36, valid_rmse = 1.5514, valid_rmse_diff = 6.8e-06
#     k =   2,     results: counter =  15, max_norm_diff = 0.33, valid_rmse = 1.5513, valid_rmse_diff = 3.2e-06
#     k =   4,     results: counter =  16, max_norm_diff = 0.14, valid_rmse = 1.5512, valid_rmse_diff = 8.3e-06
#     k =   8,     results: counter =  17, max_norm_diff = 0.20, valid_rmse = 1.5513, valid_rmse_diff = 8.4e-06
#     k =  16,     results: counter =  21, max_norm_diff = 0.21, valid_rmse = 1.5513, valid_rmse_diff = 8.8e-06
#     k =  32,     results: counter =  10, max_norm_diff = 1.50, valid_rmse = 1.5511, valid_rmse_diff = 3.5e-06
#     k =  64,     results: counter =  14, max_norm_diff = 1.13, valid_rmse = 1.5507, valid_rmse_diff = 2.2e-06
#     k = 128,     results: counter =  10, max_norm_diff = 2.19, valid_rmse = 1.5503, valid_rmse_diff = 9.9e-06
#     k = 256,     results: counter =   7, max_norm_diff = 3.80, valid_rmse = 1.5496, valid_rmse_diff = 4.1e-05
# seed = 1097
#     only biases: results: counter =  18, max_norm_diff = 0.05, valid_rmse = 1.5686, valid_rmse_diff = 8.2e-06
#     k =   1,     results: counter =  16, max_norm_diff = 0.27, valid_rmse = 1.5682, valid_rmse_diff = 9.1e-06
#     k =   2,     results: counter =  19, max_norm_diff = 0.09, valid_rmse = 1.5684, valid_rmse_diff = 8.1e-06
#     k =   4,     results: counter =  18, max_norm_diff = 0.09, valid_rmse = 1.5683, valid_rmse_diff = 8.2e-06
#     k =   8,     results: counter =  20, max_norm_diff = 0.17, valid_rmse = 1.5683, valid_rmse_diff = 8.3e-06
#     k =  16,     results: counter =  15, max_norm_diff = 0.31, valid_rmse = 1.5681, valid_rmse_diff = 6.6e-06
#     k =  32,     results: counter =  16, max_norm_diff = 0.89, valid_rmse = 1.5678, valid_rmse_diff = 9.5e-06
#     k =  64,     results: counter =  15, max_norm_diff = 1.05, valid_rmse = 1.5675, valid_rmse_diff = 2.4e-06
#     k = 128,     results: counter =  12, max_norm_diff = 1.75, valid_rmse = 1.5671, valid_rmse_diff = 1.4e-06
#     k = 256,     results: counter =  10, max_norm_diff = 2.44, valid_rmse = 1.5676, valid_rmse_diff = 5.1e-06
# seed = 1098
#     only biases: results: counter =  14, max_norm_diff = 0.13, valid_rmse = 1.5620, valid_rmse_diff = 8.3e-06
#     k =   1,     results: counter =   7, max_norm_diff = 1.37, valid_rmse = 1.5623, valid_rmse_diff = 1.9e-06
#     k =   2,     results: counter =  12, max_norm_diff = 0.31, valid_rmse = 1.5621, valid_rmse_diff = 7.9e-06
#     k =   4,     results: counter =  12, max_norm_diff = 0.22, valid_rmse = 1.5618, valid_rmse_diff = 8.4e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.21, valid_rmse = 1.5616, valid_rmse_diff = 7.9e-06
#     k =  16,     results: counter =  20, max_norm_diff = 0.23, valid_rmse = 1.5614, valid_rmse_diff = 9.2e-06
#     k =  32,     results: counter =  33, max_norm_diff = 0.40, valid_rmse = 1.5613, valid_rmse_diff = 9.8e-06
#     k =  64,     results: counter =  18, max_norm_diff = 0.84, valid_rmse = 1.5613, valid_rmse_diff = 8.4e-06
#     k = 128,     results: counter =   8, max_norm_diff = 2.86, valid_rmse = 1.5608, valid_rmse_diff = 1.8e-05
#     k = 256,     results: counter =   6, max_norm_diff = 4.61, valid_rmse = 1.5613, valid_rmse_diff = 4.4e-05
# seed = 1099
#     only biases: results: counter =  17, max_norm_diff = 0.06, valid_rmse = 1.5560, valid_rmse_diff = 1e-05
#     k =   1,     results: counter =  17, max_norm_diff = 0.13, valid_rmse = 1.5554, valid_rmse_diff = 9.5e-06
#     k =   2,     results: counter =  17, max_norm_diff = 0.15, valid_rmse = 1.5552, valid_rmse_diff = 7.6e-06
#     k =   4,     results: counter =  20, max_norm_diff = 0.09, valid_rmse = 1.5555, valid_rmse_diff = 9e-06
#     k =   8,     results: counter =  16, max_norm_diff = 0.21, valid_rmse = 1.5552, valid_rmse_diff = 9.8e-06
#     k =  16,     results: counter =  14, max_norm_diff = 0.33, valid_rmse = 1.5552, valid_rmse_diff = 9.5e-06
#     k =  32,     results: counter =  11, max_norm_diff = 1.40, valid_rmse = 1.5551, valid_rmse_diff = 1.2e-05
#     k =  64,     results: counter =  14, max_norm_diff = 1.16, valid_rmse = 1.5540, valid_rmse_diff = 6.5e-06
#     k = 128,     results: counter =  13, max_norm_diff = 1.59, valid_rmse = 1.5544, valid_rmse_diff = 8.4e-06
#     k = 256,     results: counter =   9, max_norm_diff = 2.80, valid_rmse = 1.5535, valid_rmse_diff = 3.7e-05

In [21]:
# results_biases
# array([[1.571761, 1.559215, 1.560959, 1.564176, 1.55928 , 1.56813 , 1.546814, 1.575916, 1.536698, 1.56529 , 1.56238 , 1.549472, 1.552349, 1.556605,
#         1.569076, 1.559493, 1.549275, 1.557599, 1.561112, 1.554704, 1.570371, 1.560805, 1.545088, 1.545428, 1.559383, 1.563022, 1.562683, 1.55289 ,
#         1.552897, 1.577152, 1.552395, 1.557323, 1.547916, 1.551004, 1.560356, 1.564616, 1.554963, 1.546119, 1.552234, 1.573303, 1.540693, 1.574724,
#         1.569562, 1.563625, 1.552142, 1.570709, 1.564497, 1.560122, 1.563008, 1.561975, 1.553678, 1.555846, 1.552103, 1.570898, 1.550837, 1.567096,
#         1.550066, 1.557436, 1.56706 , 1.556344, 1.541633, 1.539776, 1.574467, 1.573509, 1.556195, 1.547269, 1.550683, 1.543338, 1.544462, 1.562157,
#         1.569862, 1.559169, 1.554174, 1.5793  , 1.560522, 1.577941, 1.564521, 1.563621, 1.559991, 1.555406, 1.565422, 1.573312, 1.546358, 1.570462,
#         1.555613, 1.566235, 1.552397, 1.55734 , 1.544709, 1.551753, 1.558669, 1.550447, 1.565794, 1.560279, 1.572028, 1.569264, 1.554364, 1.56099 ,
#         1.542296, 1.559031]])

In [22]:
# results_full
# array([[1.571596, 1.55911 , 1.56073 , 1.564127, 1.559564, 1.568129, 1.546923, 1.575601, 1.536772, 1.565403, 1.562223, 1.549556, 1.552413, 1.556687,
#         1.569338, 1.559477, 1.548921, 1.557893, 1.561   , 1.554649, 1.570157, 1.560657, 1.544851, 1.545369, 1.559396, 1.562864, 1.56251 , 1.552608,
#         1.552756, 1.577113, 1.552319, 1.557059, 1.54787 , 1.550713, 1.560385, 1.564707, 1.555078, 1.546418, 1.552224, 1.572977, 1.540518, 1.574764,
#         1.57004 , 1.56399 , 1.552123, 1.570268, 1.564259, 1.560245, 1.563045, 1.562196, 1.553579, 1.556527, 1.551881, 1.570744, 1.550885, 1.566808,
#         1.54991 , 1.558576, 1.566582, 1.556335, 1.541604, 1.539596, 1.574378, 1.574319, 1.556737, 1.546943, 1.550535, 1.543455, 1.544544, 1.561964,
#         1.569988, 1.559054, 1.553992, 1.579763, 1.560321, 1.577703, 1.564521, 1.56321 , 1.559952, 1.555288, 1.565236, 1.573393, 1.54636 , 1.570616,
#         1.555241, 1.566134, 1.552685, 1.557191, 1.544526, 1.552122, 1.558795, 1.550498, 1.565618, 1.560305, 1.571998, 1.568983, 1.554095, 1.560877,
#         1.542457, 1.558924],
#        [1.571536, 1.5593  , 1.560773, 1.564088, 1.559406, 1.568348, 1.546449, 1.575643, 1.536451, 1.565609, 1.562447, 1.549036, 1.552323, 1.556818,
#         1.569382, 1.559424, 1.548338, 1.557371, 1.561101, 1.554348, 1.570251, 1.561016, 1.544701, 1.545981, 1.559255, 1.562747, 1.562959, 1.553128,
#         1.552994, 1.576854, 1.55193 , 1.557453, 1.54795 , 1.550906, 1.560219, 1.565245, 1.555148, 1.54627 , 1.551935, 1.573004, 1.540599, 1.574886,
#         1.56964 , 1.56395 , 1.551905, 1.570605, 1.564023, 1.560106, 1.563312, 1.562526, 1.55348 , 1.555996, 1.551608, 1.570762, 1.550706, 1.566696,
#         1.55006 , 1.558849, 1.566332, 1.555982, 1.541634, 1.539262, 1.574045, 1.574587, 1.556414, 1.54737 , 1.550633, 1.543038, 1.544212, 1.561853,
#         1.569807, 1.559137, 1.553966, 1.579632, 1.560435, 1.577892, 1.56401 , 1.563426, 1.559826, 1.555168, 1.565524, 1.573759, 1.546432, 1.570403,
#         1.55555 , 1.565863, 1.552503, 1.557405, 1.544658, 1.552175, 1.558958, 1.550275, 1.565451, 1.560041, 1.571611, 1.56899 , 1.554185, 1.561211,
#         1.54193 , 1.559271],
#        [1.571606, 1.558955, 1.560734, 1.564066, 1.559033, 1.568059, 1.546352, 1.575722, 1.536524, 1.565218, 1.562248, 1.549024, 1.552299, 1.556447,
#         1.568766, 1.559372, 1.548898, 1.557179, 1.560918, 1.554494, 1.57019 , 1.560546, 1.544921, 1.545432, 1.559239, 1.562654, 1.562628, 1.552712,
#         1.552667, 1.577071, 1.552113, 1.557278, 1.547804, 1.550768, 1.560119, 1.564484, 1.554667, 1.545665, 1.552001, 1.572992, 1.540546, 1.574685,
#         1.569492, 1.563505, 1.551881, 1.57033 , 1.564287, 1.560127, 1.562829, 1.561837, 1.553517, 1.555584, 1.551867, 1.570725, 1.550782, 1.566875,
#         1.549947, 1.557333, 1.566673, 1.55604 , 1.54156 , 1.53949 , 1.57423 , 1.57336 , 1.556057, 1.546896, 1.550416, 1.54306 , 1.54429 , 1.562006,
#         1.569755, 1.559035, 1.5539  , 1.579136, 1.56022 , 1.577668, 1.564315, 1.563342, 1.559887, 1.555236, 1.565334, 1.573198, 1.546284, 1.570223,
#         1.555345, 1.566089, 1.552041, 1.557267, 1.544485, 1.551655, 1.558472, 1.550402, 1.565451, 1.560123, 1.571731, 1.569035, 1.554121, 1.560837,
#         1.542084, 1.558955],
#        [1.571302, 1.55863 , 1.560667, 1.563893, 1.558873, 1.568   , 1.546039, 1.575691, 1.536432, 1.564963, 1.562141, 1.548662, 1.552164, 1.556456,
#         1.568827, 1.559113, 1.548462, 1.557232, 1.560657, 1.554453, 1.569977, 1.560435, 1.544806, 1.545107, 1.558965, 1.562496, 1.562668, 1.552653,
#         1.552516, 1.576932, 1.551892, 1.55702 , 1.548025, 1.550479, 1.560317, 1.564212, 1.554559, 1.545844, 1.551922, 1.572749, 1.540499, 1.574476,
#         1.569357, 1.563281, 1.551739, 1.570258, 1.564126, 1.559933, 1.562693, 1.561616, 1.553442, 1.555379, 1.551907, 1.57055 , 1.550616, 1.566727,
#         1.549776, 1.557189, 1.566252, 1.556063, 1.541435, 1.539372, 1.574039, 1.573305, 1.555889, 1.546914, 1.550302, 1.542845, 1.544201, 1.561829,
#         1.569688, 1.55895 , 1.553661, 1.578745, 1.56028 , 1.577539, 1.564222, 1.563174, 1.559883, 1.555127, 1.565011, 1.573336, 1.546369, 1.569898,
#         1.555357, 1.565866, 1.55191 , 1.557191, 1.544402, 1.551793, 1.558261, 1.550461, 1.565226, 1.559864, 1.571347, 1.568729, 1.554008, 1.560652,
#         1.541803, 1.558991],
#        [1.571418, 1.558838, 1.560471, 1.563814, 1.558422, 1.567768, 1.546001, 1.575817, 1.53633 , 1.564902, 1.562206, 1.54871 , 1.552269, 1.556048,
#         1.568272, 1.559004, 1.548319, 1.556588, 1.560876, 1.554281, 1.569754, 1.560511, 1.544686, 1.545188, 1.558727, 1.562388, 1.562556, 1.552611,
#         1.552438, 1.576912, 1.551889, 1.556873, 1.547886, 1.55054 , 1.560218, 1.564054, 1.554393, 1.545396, 1.551653, 1.57239 , 1.540362, 1.574434,
#         1.569395, 1.563232, 1.551613, 1.569941, 1.563812, 1.559681, 1.562506, 1.56175 , 1.553269, 1.55541 , 1.551643, 1.570448, 1.550683, 1.566732,
#         1.549561, 1.557139, 1.566015, 1.555931, 1.541535, 1.539252, 1.57373 , 1.57334 , 1.555738, 1.546948, 1.550298, 1.542617, 1.544178, 1.561612,
#         1.56971 , 1.558652, 1.553476, 1.578711, 1.559723, 1.577557, 1.56384 , 1.563129, 1.559665, 1.555114, 1.564609, 1.573292, 1.546061, 1.569645,
#         1.554873, 1.565911, 1.551886, 1.557078, 1.544003, 1.551712, 1.558165, 1.550398, 1.565689, 1.559815, 1.571303, 1.56859 , 1.553634, 1.560387,
#         1.541445, 1.559   ],
#        [1.571537, 1.557756, 1.560564, 1.564644, 1.557936, 1.56581 , 1.545147, 1.576795, 1.536938, 1.564835, 1.561089, 1.547066, 1.551558, 1.554538,
#         1.568166, 1.557217, 1.548598, 1.556433, 1.560105, 1.555009, 1.568285, 1.560214, 1.543293, 1.544762, 1.558228, 1.562059, 1.562848, 1.552095,
#         1.551311, 1.576332, 1.551102, 1.55608 , 1.547411, 1.549475, 1.560541, 1.562425, 1.55469 , 1.544376, 1.551373, 1.572952, 1.540424, 1.574086,
#         1.5701  , 1.562861, 1.552003, 1.569266, 1.564915, 1.559418, 1.562823, 1.562547, 1.552785, 1.555223, 1.552166, 1.570211, 1.551287, 1.565579,
#         1.548868, 1.557068, 1.565481, 1.553671, 1.540614, 1.539215, 1.573351, 1.572524, 1.555801, 1.546216, 1.550471, 1.542281, 1.544077, 1.561935,
#         1.569659, 1.558605, 1.552496, 1.576963, 1.559583, 1.576361, 1.563222, 1.563068, 1.559364, 1.555378, 1.564404, 1.572798, 1.54549 , 1.569411,
#         1.553935, 1.565099, 1.550597, 1.556803, 1.543812, 1.551739, 1.557259, 1.550794, 1.56488 , 1.559544, 1.571313, 1.567655, 1.553572, 1.559744,
#         1.539868, 1.559092],
#        [1.571349, 1.557806, 1.559652, 1.562842, 1.557134, 1.565895, 1.544645, 1.576587, 1.535522, 1.565227, 1.560839, 1.547198, 1.551296, 1.554185,
#         1.567435, 1.556912, 1.548137, 1.555663, 1.559402, 1.552857, 1.567712, 1.55975 , 1.542762, 1.544658, 1.55734 , 1.561289, 1.562468, 1.551468,
#         1.551432, 1.575661, 1.550978, 1.555473, 1.5474  , 1.549883, 1.559988, 1.56248 , 1.554335, 1.543771, 1.550792, 1.571475, 1.538989, 1.57423 ,
#         1.569716, 1.562678, 1.552076, 1.568801, 1.565025, 1.559251, 1.562412, 1.562327, 1.552374, 1.554256, 1.552004, 1.570254, 1.55072 , 1.565449,
#         1.548676, 1.556386, 1.565482, 1.553292, 1.540558, 1.53943 , 1.57319 , 1.572975, 1.555285, 1.546369, 1.550493, 1.542647, 1.54432 , 1.561497,
#         1.569969, 1.557851, 1.552487, 1.576552, 1.559461, 1.576422, 1.562749, 1.562521, 1.559544, 1.555093, 1.565187, 1.571988, 1.544968, 1.568685,
#         1.554064, 1.564361, 1.551307, 1.556482, 1.543338, 1.551641, 1.557396, 1.549778, 1.565247, 1.55837 , 1.570968, 1.56744 , 1.552309, 1.560203,
#         1.54026 , 1.558406],
#        [1.570714, 1.55714 , 1.560385, 1.562198, 1.556829, 1.566076, 1.544729, 1.576154, 1.535614, 1.564575, 1.5611  , 1.546311, 1.55039 , 1.553676,
#         1.567649, 1.556696, 1.547765, 1.555545, 1.559535, 1.552952, 1.567979, 1.559458, 1.542731, 1.543799, 1.557153, 1.560922, 1.561649, 1.551215,
#         1.550976, 1.574492, 1.550347, 1.554654, 1.547205, 1.549203, 1.559512, 1.562039, 1.555018, 1.543596, 1.550403, 1.571064, 1.538869, 1.573935,
#         1.569139, 1.563388, 1.551385, 1.568949, 1.563904, 1.558081, 1.562367, 1.562737, 1.551399, 1.55379 , 1.550942, 1.569945, 1.55026 , 1.565157,
#         1.548763, 1.555956, 1.565195, 1.553464, 1.540796, 1.538551, 1.572671, 1.574023, 1.553932, 1.545603, 1.549199, 1.541933, 1.543785, 1.561093,
#         1.569243, 1.557794, 1.552225, 1.576371, 1.558418, 1.575995, 1.562633, 1.562448, 1.55921 , 1.554533, 1.56444 , 1.57252 , 1.545404, 1.568148,
#         1.553456, 1.563389, 1.550514, 1.556135, 1.542445, 1.551768, 1.556483, 1.548748, 1.564704, 1.557549, 1.570523, 1.565908, 1.552428, 1.558814,
#         1.540551, 1.558943],
#        [1.569899, 1.556734, 1.559845, 1.561504, 1.555712, 1.565676, 1.54393 , 1.576272, 1.535125, 1.564131, 1.560951, 1.546287, 1.550202, 1.552955,
#         1.567251, 1.557123, 1.54724 , 1.556307, 1.559287, 1.552664, 1.567693, 1.558832, 1.54259 , 1.544625, 1.557275, 1.560346, 1.561505, 1.55033 ,
#         1.549902, 1.574403, 1.54997 , 1.553956, 1.54725 , 1.549447, 1.559453, 1.561623, 1.554502, 1.543673, 1.54996 , 1.569816, 1.538379, 1.573232,
#         1.568983, 1.561924, 1.551076, 1.568732, 1.563977, 1.557683, 1.561896, 1.561738, 1.551295, 1.553154, 1.55123 , 1.569879, 1.549339, 1.56485 ,
#         1.548112, 1.555484, 1.564427, 1.55309 , 1.539908, 1.537859, 1.572268, 1.57313 , 1.553976, 1.545964, 1.548998, 1.54143 , 1.542918, 1.560314,
#         1.569047, 1.557922, 1.552075, 1.576044, 1.558802, 1.575755, 1.56257 , 1.563134, 1.559181, 1.554584, 1.564173, 1.57271 , 1.545263, 1.568191,
#         1.553694, 1.563188, 1.550209, 1.555973, 1.542767, 1.55111 , 1.556408, 1.549369, 1.564223, 1.557348, 1.569559, 1.566951, 1.551927, 1.558487,
#         1.539768, 1.558036]])